# Urban Flood — M2 NT1: V2 Top-3 DL Refit + Inference


Refit top-3 V2 DL models from experiment results using GroupKFold on full train+valid data.


**Models:**

1. N2_transformer_scse (RMSE=0.327484, best)

2. N5_big_tf_w200 (RMSE=0.328257)

3. N3_transformer_gcn (RMSE=0.344853)


**Output:** 3 parquet files with `model_id, event_id, node_type, node_id, water_level`

In [1]:
from __future__ import annotations

import gc
import os
import re
import time
import copy
import random
import warnings
import ctypes
from collections import defaultdict
from pathlib import Path

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh

import numpy as np
import pandas as pd
import pandas.api.types
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

In [2]:
# =========================================================

# CONFIG

# =========================================================

DATA_ROOT = Path("/kaggle/input/datasets/bulbazavril/fullds/Models")


KEY_COLS = ["model_id", "event_id", "node_type", "node_id"]

TARGET_COL = "water_level"

SUBMISSION_COLS = ["row_id"] + KEY_COLS + [TARGET_COL]


VALID_EVENTS_PER_MODEL = 15

RANDOM_SEED = 42

MIN_PRED_TIMESTEP = 10

WARMUP_STEPS = 10

USE_TARGET_DELTA = True

USE_TARGET_RATIO = False   # v8: WL/node_scale ratio target (experimental)


# --- Experimental flags ---

USE_SPECTRAL_EMBED = False   # Laplacian eigenvector embeddings

SPECTRAL_DIM = 8             # Number of eigenvectors

USE_AUX_TARGETS = False      # Predict inlet_flow → use as feature


# --- CatBoost seeds ---

CB_VALID_SEED = 56

CB_REFIT_SEEDS = [56]


# --- BiGRU seeds ---

BIGRU_VALID_SEED = 56

BIGRU_REFIT_SEEDS = [56]

BIGRU_GROUPKFOLD_SPLITS = 5


# --- CatBoost params from source notebook ---

CB_PARAMS = {

    (2, 1): {

        'loss_function': 'RMSE',

        'iterations': 50_000,

        'max_depth': 8,

        'learning_rate': 0.02,

        'l2_leaf_reg': 15,

        'min_data_in_leaf': 100,

        'border_count': 254,

        'eval_metric': 'RMSE',

        'task_type': 'GPU',

        'random_seed': 56,

        'verbose': 500,

    },

}


# --- DL base params (H100 optimized) ---

DL_BASE_PARAMS = {

    'tab_hidden': 256,

    'head_hidden': 256,

    'dropout': 0.15,

    'lr': 2e-4,

    'weight_decay': 1e-5,

    'epochs': 30,

    'patience': 6,

    'batch_size': 2048,

    'eval_batch_size': 16384,

    'num_workers': 0,

    'grad_clip': 1.0,

    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

}


# --- DL experiments ---

DL_EXPERIMENTS = {

    'N5_big_tf_w200': {

        **DL_BASE_PARAMS,

        'arch': 'transformer_prenorm',

        'd_model': 64,

        'nhead': 4,

        'tf_layers': 2,

        'tf_ff_mult': 2,

        'use_rain_window': True,

        'past_window': 200,

        'future_window': 10,

        'use_num_embed': False,

        'use_scse': False,

        'tab_hidden': 256,

        'head_hidden': 256,

        'lr': 1.5e-4,

        'epochs': 40,

        'patience': 8,

        'batch_size': 1024,

    },

    'N2_transformer_scse': {

        **DL_BASE_PARAMS,

        'arch': 'transformer_prenorm',

        'd_model': 64,

        'nhead': 4,

        'tf_layers': 2,

        'tf_ff_mult': 2,

        'use_rain_window': True,

        'past_window': 120,

        'future_window': 10,

        'use_num_embed': False,

        'use_scse': True,

        'scse_reduction': 4,

        'tab_hidden': 256,

        'head_hidden': 256,

        'lr': 1.5e-4,

        'epochs': 40,

        'patience': 8,

        'batch_size': 1536,

    },

    'N3_transformer_gcn': {

        **DL_BASE_PARAMS,

        'arch': 'transformer_prenorm',

        'd_model': 64,

        'nhead': 4,

        'tf_layers': 2,

        'tf_ff_mult': 2,

        'use_rain_window': True,

        'past_window': 120,

        'future_window': 10,

        'use_num_embed': False,

        'use_scse': False,

        'use_gcn': True,

        'gcn_hidden': 64,

        'gcn_out': 32,

        'gcn_layers': 2,

        'gcn_epochs': 100,

        'tab_hidden': 256,

        'head_hidden': 256,

        'lr': 1.5e-4,

        'epochs': 40,

        'patience': 8,

        'batch_size': 1536,

    },

}


# --- BiGRU params (legacy compat for old code paths) ---

BIGRU_PARAMS = {

    (2, 1): {

        'seq_hidden': 96,

        'seq_layers': 2,

        'tab_hidden': 256,

        'head_hidden': 256,

        'dropout': 0.15,

        'lr': 2e-4,

        'weight_decay': 1e-5,

        'epochs': 30,

        'patience': 6,

        'batch_size': 2048,

        'eval_batch_size': 16384,

        'num_workers': 0,

        'grad_clip': 1.0,

        'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    },

}


FINAL_TEST_PRED_PARQUET_PATH = Path("submission_m2_nt1_refit_cb_bigru_ensemble.parquet")

FINAL_TEST_COMPONENTS_PARQUET_PATH = Path("submission_m2_nt1_refit_cb_bigru_components.parquet")


print("Config ready: CatBoost + BiGRU ensemble, 1 validation seed per model, BiGRU test-time ensemble via GroupKFold folds with per-fold best epochs, parquet export.")

Config ready: CatBoost + BiGRU ensemble, 1 validation seed per model, BiGRU test-time ensemble via GroupKFold folds with per-fold best epochs, parquet export.


In [3]:
# =========================================================
# METRIC
# =========================================================
STD_DEV_DICT = {
    (1, 1): 16.877747, (1, 2): 14.378797,
    (2, 1): 3.191784,  (2, 2): 2.727131,
}

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def local_score(solution_df, submission_df):
    sol = solution_df.copy()
    sub = submission_df.copy()
    model_scores = []
    for mid in sorted(sol['model_id'].unique()):
        nt_raw = {1: [], 2: []}
        event_scores = []
        for evid in sorted(sol[sol['model_id']==mid]['event_id'].unique()):
            nt_scores = []
            for nt in [1, 2]:
                mask_s = (sol['model_id']==mid) & (sol['event_id']==evid) & (sol['node_type']==nt)
                mask_b = (sub['model_id']==mid) & (sub['event_id']==evid) & (sub['node_type']==nt)
                if mask_s.sum() == 0: continue
                s_df = sol[mask_s]; b_df = sub[mask_b]
                sd = STD_DEV_DICT.get((mid, nt), 1.0)
                node_std_rmses = []
                for nid in s_df['node_id'].unique():
                    nm_s = s_df[s_df['node_id']==nid]['water_level'].values
                    nm_b = b_df[b_df['node_id']==nid]['water_level'].values
                    if len(nm_s) > 1 and len(nm_s) == len(nm_b):
                        r = rmse(nm_s, nm_b)
                        node_std_rmses.append(r / sd)
                        nt_raw[nt].append(r)
                if node_std_rmses:
                    nt_scores.append(np.mean(node_std_rmses))
            if nt_scores:
                event_scores.append(np.mean(nt_scores))
        if event_scores:
            ms = np.mean(event_scores)
            model_scores.append(ms)
            print(f"  Model {mid}: Std RMSE = {ms:.6f} ({len(event_scores)} events)")
            for nt in [1, 2]:
                if nt_raw[nt]:
                    print(f"    NT{nt}: raw RMSE mean={np.mean(nt_raw[nt]):.6f}")
    final = np.mean(model_scores)
    print(f"  Overall: {final:.6f}")
    return final

In [4]:
# =========================================================
# UTILITIES
# =========================================================
def _free_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

def ncol(col):
    col = str(col).strip().lower()
    col = col.replace('\u00b2', '2').replace('\u00b3', '3').replace('%', 'pct')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return re.sub(r'_+', '_', col).strip('_')

def read_csv(path):
    df = pd.read_csv(path)
    df.columns = [ncol(c) for c in df.columns]
    drop = [c for c in df.columns if c.startswith('unnamed')]
    if drop: df = df.drop(columns=drop)
    return df

def read_table(path):
    if str(path).endswith('.parquet'): df = pd.read_parquet(path)
    else: df = pd.read_csv(path)
    df.columns = [ncol(c) for c in df.columns]
    return df

def fcol(columns, names=(), tokens=(), required=True, label='col'):
    cols = list(columns)
    for n in names:
        if n in cols: return n
    for c in cols:
        if all(t in c for t in tokens): return c
    if required: raise KeyError(f"Cannot find {label} in {cols}")
    return None

def ensure_nid(df):
    c = fcol(df.columns, ('node_idx','node_id','id'), ('node',), False, 'nid')
    out = df.copy()
    if c is None: out['node_id'] = np.arange(len(out), dtype='int64')
    elif c != 'node_id': out = out.rename(columns={c: 'node_id'})
    out['node_id'] = out['node_id'].astype('int64')
    return out

def eid(d): return int(d.name.split('_')[1])
def mid_fn(d): return int(d.name.split('_')[1])
def event_dirs(split_dir):
    return sorted([d for d in split_dir.glob('event_*') if d.is_dir()], key=eid)

def dyn_path(event_dir, nt):
    return event_dir / f"{'1d' if nt==1 else '2d'}_nodes_dynamic_all.csv"

def edge_dyn_path(event_dir, nt):
    return event_dir / f"{'1d' if nt==1 else '2d'}_edges_dynamic_all.csv"

def find_file(model_dir, filename):
    for s in ('train', 'test'):
        p = model_dir / s / filename
        if p.exists(): return p
    return None

def downcast_float32(df):
    for c in df.columns:
        if df[c].dtype == 'float64':
            df[c] = df[c].astype('float32')
    return df

print("Utilities ready.")

Utilities ready.


In [5]:
# =========================================================
# STATIC NODE FEATURES
# =========================================================
STATIC_CACHE = {}

def load_static(model_id, node_type):
    k = (model_id, node_type)
    if k in STATIC_CACHE: return STATIC_CACHE[k]
    md = DATA_ROOT / f'Model_{model_id}'
    if node_type == 1:
        cands = ['1d_nodes_static.csv']
    else:
        cands = ['2d_nodes_static.csv', '2d_nodes_index.csv']
    for fn in cands:
        p = find_file(md, fn)
        if p:
            df = read_csv(p)
            df = ensure_nid(df)
            df = df.drop_duplicates('node_id').reset_index(drop=True)
            STATIC_CACHE[k] = df
            return df
    raise FileNotFoundError(f'Static not found for m{model_id} nt{node_type}')

print("Static loader ready.")

Static loader ready.


In [6]:
# =========================================================
# GRAPH FEATURES + MULTI-HOP + CENTRALITY + PIPE CAPACITY
# =========================================================
GRAPH_CACHE = {}
ADJ_CACHE = {}
EDGE_INDEX_CACHE = {}

def _count_upstream_total(upstream_dict, all_nids):
    cache = {}
    def _get_ancestors(nid, visited):
        if nid in cache: return cache[nid]
        if nid in visited: return set()
        visited.add(nid)
        result = set()
        for parent in upstream_dict.get(nid, []):
            result.add(parent)
            result.update(_get_ancestors(parent, visited))
        cache[nid] = result
        return result
    out = {}
    for nid in all_nids:
        out[int(nid)] = len(_get_ancestors(int(nid), set()))
    return out

def _compute_2hop_features(adj, node_feat_map, all_nids, feat_name):
    """Compute 2-hop neighbor aggregation for a given feature."""
    result = {}
    for nid in all_nids:
        nid = int(nid)
        hop1 = adj.get(nid, [])
        hop2_set = set()
        for n1 in hop1:
            for n2 in adj.get(n1, []):
                if n2 != nid and n2 not in hop1:
                    hop2_set.add(n2)
        if hop2_set:
            vals = [node_feat_map.get(n, np.nan) for n in hop2_set]
            vals = [v for v in vals if not np.isnan(v)]
            if vals:
                result[nid] = {
                    f'hop2_{feat_name}_mean': np.mean(vals),
                    f'hop2_{feat_name}_max': np.max(vals),
                    f'hop2_{feat_name}_min': np.min(vals),
                    f'hop2_count': len(hop2_set),
                }
    return result

def _pipe_capacity_manning(diameter, roughness, slope):
    """Manning's formula for full-pipe capacity: Q = (1/n) * A * R^(2/3) * S^(1/2).
    Circular pipe: A = pi*d^2/4, R = d/4.
    """
    if diameter <= 0 or roughness <= 0 or slope <= 0:
        return 0.0
    area = np.pi * diameter**2 / 4.0
    hyd_radius = diameter / 4.0
    return (1.0 / roughness) * area * hyd_radius**(2.0/3.0) * np.sqrt(slope)

def load_graph_feats(model_id, node_type):
    k = (model_id, node_type)
    if k in GRAPH_CACHE: return GRAPH_CACHE[k]
    md = DATA_ROOT / f'Model_{model_id}'
    prefix = '1d' if node_type == 1 else '2d'
    ei_path = find_file(md, f'{prefix}_edge_index.csv')

    if ei_path is None:
        print(f"  [WARN] {prefix}_edge_index.csv not found for Model_{model_id}")
        GRAPH_CACHE[k] = pd.DataFrame()
        return GRAPH_CACHE[k]

    ei = read_csv(ei_path)
    print(f"  Model {model_id} {prefix} edge_index: {len(ei)} edges, cols={list(ei.columns)}")

    cols = list(ei.columns)
    fc = fcol(cols, ('from_node','from_node_id','source'), ('from',), False, 'from')
    tc = fcol(cols, ('to_node','to_node_id','target'), ('to',), False, 'to')
    if fc is None or tc is None: fc, tc = cols[0], cols[1]

    EDGE_INDEX_CACHE[k] = (fc, tc, ei)

    adj = defaultdict(set)
    downstream = defaultdict(list)
    upstream = defaultdict(list)
    for _, r in ei.iterrows():
        u, v = int(r[fc]), int(r[tc])
        adj[u].add(v); adj[v].add(u)
        downstream[u].append(v)
        upstream[v].append(u)
    adj = {k_: list(v) for k_, v in adj.items()}
    ADJ_CACHE[k] = adj

    # Load edge static features
    es_path = find_file(md, f'{prefix}_edges_static.csv')
    edge_static = read_csv(es_path) if es_path else None
    if edge_static is not None:
        print(f"    Edge static: {len(edge_static)} edges, cols={list(edge_static.columns)}")

    efmap = {}
    if edge_static is not None and len(ei) == len(edge_static):
        for i in range(len(ei)):
            u, v = int(ei.iloc[i][fc]), int(ei.iloc[i][tc])
            efmap[(u, v)] = edge_static.iloc[i].to_dict()

    # 1D-2D connections
    conn = find_file(md, '1d2d_connections.csv')
    conn_map = defaultdict(list)
    conn_1d_to_2d = defaultdict(list)
    if conn:
        cdf = read_csv(conn)
        cc = list(cdf.columns)
        c1 = fcol(cc, ('1d_node_id','node_1d'), ('1d',), False, '1d')
        c2 = fcol(cc, ('2d_node_id','node_2d'), ('2d',), False, '2d')
        if c1 is None or c2 is None: c1, c2 = cc[0], cc[1]
        for _, r in cdf.iterrows():
            n1, n2 = int(r[c1]), int(r[c2])
            conn_1d_to_2d[n1].append(n2)
            if node_type == 1: conn_map[n1].append(n2)
            else: conn_map[n2].append(n1)

    static = load_static(model_id, node_type)
    all_nids = static['node_id'].unique()

    # Transitive upstream count (1D only)
    upstream_total = {}
    if node_type == 1:
        upstream_total = _count_upstream_total(upstream, all_nids)
        print(f"    Upstream total: max={max(upstream_total.values()) if upstream_total else 0}")

    # Elevation maps for gradients + 2-hop
    elev_map = {}
    if node_type == 2:
        elev_c = fcol(static.columns, ('elevation','min_elevation','centroid_elevation'),
                      ('elev',), False, 'elev')
        if elev_c:
            elev_map = dict(zip(static['node_id'].astype(int), static[elev_c].astype(float)))
    elif node_type == 1:
        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')
        if inv_c:
            elev_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))

    # Enhanced coupling: elevation difference 1D<->2D
    coupling_elev_diff = {}
    if node_type == 1 and conn_1d_to_2d:
        s2d = load_static(model_id, 2)
        elev_c_2d = fcol(s2d.columns, ('elevation','min_elevation','centroid_elevation'),
                         ('elev',), False, 'elev')
        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')
        if elev_c_2d and inv_c:
            elev_2d_map = dict(zip(s2d['node_id'].astype(int), s2d[elev_c_2d].astype(float)))
            inv_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))
            for n1d, n2d_list in conn_1d_to_2d.items():
                e2d_vals = [elev_2d_map.get(n2, np.nan) for n2 in n2d_list]
                e2d_clean = [v for v in e2d_vals if not np.isnan(v)]
                inv_val = inv_map.get(n1d, np.nan)
                if e2d_clean and not np.isnan(inv_val):
                    coupling_elev_diff[n1d] = np.mean(e2d_clean) - inv_val

    # v5.5: Graph centrality via NetworkX (fast for small 1D graphs, ok for 2D)
    centrality_pr = {}
    centrality_deg = {}
    try:
        import networkx as nx
        G = nx.Graph()
        for _, r in ei.iterrows():
            G.add_edge(int(r[fc]), int(r[tc]))
        centrality_pr = nx.pagerank(G, max_iter=100, tol=1e-4)
        centrality_deg = nx.degree_centrality(G)
        print(f"    Centrality: PageRank + degree for {G.number_of_nodes()} nodes")
    except Exception as e:
        print(f"    [WARN] centrality failed: {e}")

    # v5.5: 2-hop features
    hop2_feats = {}
    if elev_map:
        hop2_feats = _compute_2hop_features(adj, elev_map, all_nids, 'elev')
        print(f"    2-hop elev features: {len(hop2_feats)} nodes")

    # v5.5: Pipe capacity (1D only, Manning's formula)
    pipe_cap_map = {}
    if node_type == 1 and efmap:
        node_cap_in = defaultdict(list)
        node_cap_out = defaultdict(list)
        for (u, v), ef in efmap.items():
            diam = float(ef.get('diameter', 0) or 0)
            rough = float(ef.get('roughness', 0) or 0)
            slp = abs(float(ef.get('slope', 0) or 0))
            cap = _pipe_capacity_manning(diam, rough, slp)
            node_cap_out[u].append(cap)
            node_cap_in[v].append(cap)
        for nid in all_nids:
            nid = int(nid)
            caps_in = node_cap_in.get(nid, [])
            caps_out = node_cap_out.get(nid, [])
            all_caps = caps_in + caps_out
            if all_caps:
                pipe_cap_map[nid] = {
                    'pipe_cap_total_in': sum(caps_in),
                    'pipe_cap_total_out': sum(caps_out),
                    'pipe_cap_max': max(all_caps),
                    'pipe_cap_mean': np.mean(all_caps),
                }
        print(f"    Pipe capacity: {len(pipe_cap_map)} nodes")

    recs = []
    for nid in all_nids:
        nid = int(nid)
        r = {'node_id': nid, 'degree': len(adj.get(nid, []))}

        # Centrality
        r['pagerank'] = centrality_pr.get(nid, 0.0)
        r['degree_centrality'] = centrality_deg.get(nid, 0.0)

        # 2-hop features
        if nid in hop2_feats:
            r.update(hop2_feats[nid])

        if node_type == 1:
            up = upstream.get(nid, [])
            dn = downstream.get(nid, [])
            r['n_upstream'] = len(up)
            r['n_downstream'] = len(dn)
            r['upstream_total'] = upstream_total.get(nid, 0)

            # Aggregate upstream/downstream edge static features
            up_feats = [efmap.get((u, nid), {}) for u in up]
            dn_feats = [efmap.get((nid, v), {}) for v in dn]
            for pname, flist in [('up', up_feats), ('dn', dn_feats)]:
                if flist:
                    feat_names = []
                    for fname in flist[0].keys():
                        try:
                            vals = [float(f.get(fname, np.nan)) for f in flist if pd.notna(f.get(fname, np.nan))]
                        except Exception:
                            vals = []
                        if vals:
                            feat_names.append(fname)
                    for fname in feat_names:
                        vals = []
                        for f in flist:
                            try:
                                v = float(f.get(fname, np.nan))
                                if pd.notna(v):
                                    vals.append(v)
                            except Exception:
                                pass
                        if vals:
                            arr = np.asarray(vals, dtype='float64')
                            r[f'{pname}_{fname}_mean'] = np.mean(arr)
                            r[f'{pname}_{fname}_min'] = np.min(arr)
                            r[f'{pname}_{fname}_max'] = np.max(arr)
                            r[f'{pname}_{fname}_std'] = np.std(arr) if arr.size > 1 else 0.0

            conn_nodes = conn_map.get(nid, [])
            r['has_2d_conn'] = int(len(conn_nodes) > 0)
            r['n_2d_conn'] = len(conn_nodes)
            r['coupling_elev_diff'] = coupling_elev_diff.get(nid, np.nan)

            # Pipe capacity
            if nid in pipe_cap_map:
                r.update(pipe_cap_map[nid])

        else:  # 2D
            conn_nodes = conn_map.get(nid, [])
            r['has_1d_conn'] = int(len(conn_nodes) > 0)
            r['n_1d_conn'] = len(conn_nodes)
            nbrs = adj.get(nid, [])
            if nbrs and elev_map:
                own_elev = elev_map.get(nid, np.nan)
                nbr_elevs = [elev_map.get(n, np.nan) for n in nbrs]
                nbr_elevs_clean = [e for e in nbr_elevs if not np.isnan(e)]
                if nbr_elevs_clean:
                    r['nbr_mean_elev'] = np.mean(nbr_elevs_clean)
                    r['nbr_min_elev'] = np.min(nbr_elevs_clean)
                    if not np.isnan(own_elev):
                        r['elev_gradient_mean'] = own_elev - np.mean(nbr_elevs_clean)
                        r['elev_gradient_max'] = own_elev - np.min(nbr_elevs_clean)

            # v5: aggregate 2D static edge features to nodes
            if efmap:
                nbr_edge_feats = []
                for nb in nbrs:
                    ef = efmap.get((nid, nb)) or efmap.get((nb, nid))
                    if ef: nbr_edge_feats.append(ef)
                if nbr_edge_feats:
                    for fname in list(nbr_edge_feats[0].keys())[:6]:
                        vals = []
                        for f in nbr_edge_feats:
                            try: vals.append(float(f.get(fname, 0)))
                            except: pass
                        if vals:
                            r[f'nbr_{fname}_mean'] = np.mean(vals)
        recs.append(r)

    result = pd.DataFrame(recs)
    result['node_id'] = result['node_id'].astype('int64')
    print(f"    Graph features: {len(result)} nodes, {len(result.columns)-1} features")
    GRAPH_CACHE[k] = result
    return result

# --- v6 Experimental: Spectral graph embeddings ---
SPECTRAL_CACHE = {}

def compute_spectral_embeddings(model_id, node_type):
    """Compute Laplacian eigenvector embeddings from graph structure."""
    if not USE_SPECTRAL_EMBED:
        return pd.DataFrame()
    k = (model_id, node_type)
    if k in SPECTRAL_CACHE:
        return SPECTRAL_CACHE[k]

    try:
        ei = load_edge_index(model_id, node_type)
        if ei is None or len(ei) == 0:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        src_col = fcol(ei.columns, ('source','from','src'), ('node','id'), label='src')
        dst_col = fcol(ei.columns, ('target','to','dst','dest'), ('node','id'), label='dst')
        if src_col is None or dst_col is None:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        nodes = sorted(set(ei[src_col].tolist() + ei[dst_col].tolist()))
        n = len(nodes)
        if n < SPECTRAL_DIM + 2:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        nid_map = {v: i for i, v in enumerate(nodes)}
        rows = [nid_map[v] for v in ei[src_col]]
        cols = [nid_map[v] for v in ei[dst_col]]
        # Symmetric adjacency
        rows_full = rows + cols
        cols_full = cols + rows
        vals = np.ones(len(rows_full), dtype='float32')
        A = csr_matrix((vals, (rows_full, cols_full)), shape=(n, n))
        # Remove self-loops, ensure binary
        A.setdiag(0)
        A.eliminate_zeros()
        A.data[:] = 1.0

        # Laplacian: L = D - A
        deg = np.array(A.sum(axis=1)).flatten()
        D = csr_matrix((deg, (range(n), range(n))), shape=(n, n))
        L = D - A

        # Normalized Laplacian for stability
        deg_inv_sqrt = np.zeros(n, dtype='float64')
        mask = deg > 0
        deg_inv_sqrt[mask] = 1.0 / np.sqrt(deg[mask])
        D_inv_sqrt = csr_matrix((deg_inv_sqrt, (range(n), range(n))), shape=(n, n))
        L_norm = D_inv_sqrt @ L @ D_inv_sqrt

        # Smallest non-trivial eigenvectors
        n_eig = min(SPECTRAL_DIM + 1, n - 1)
        eigenvalues, eigenvectors = eigsh(L_norm.astype('float64'), k=n_eig, which='SM')

        # Skip first eigenvector (constant), take next SPECTRAL_DIM
        emb = eigenvectors[:, 1:SPECTRAL_DIM+1].astype('float32')
        result = pd.DataFrame(emb, columns=[f'spectral_{i}' for i in range(emb.shape[1])])
        result['node_id'] = nodes
        SPECTRAL_CACHE[k] = result
        print(f"    Spectral embeddings: {emb.shape[1]} dims for {n} nodes")
        return result

    except Exception as e:
        print(f"    Spectral embeddings failed: {e}")
        SPECTRAL_CACHE[k] = pd.DataFrame()
        return pd.DataFrame()

print("Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).")

Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).


In [7]:
# =========================================================
# DIST TO NEAREST DRAIN (for 2D nodes)
# =========================================================
DIST_DRAIN_CACHE = {}

def load_dist_to_drain(model_id):
    if model_id in DIST_DRAIN_CACHE: return DIST_DRAIN_CACHE[model_id]
    try:
        s1d = load_static(model_id, 1)
        s2d = load_static(model_id, 2)
        x1 = fcol(s1d.columns, ('position_x',), ('position','x'), False, 'x')
        y1 = fcol(s1d.columns, ('position_y',), ('position','y'), False, 'y')
        x2 = fcol(s2d.columns, ('position_x',), ('position','x'), False, 'x')
        y2 = fcol(s2d.columns, ('position_y',), ('position','y'), False, 'y')
        if not all([x1, y1, x2, y2]):
            DIST_DRAIN_CACHE[model_id] = pd.DataFrame()
            return DIST_DRAIN_CACHE[model_id]
        from scipy.spatial import cKDTree
        tree = cKDTree(s1d[[x1, y1]].values.astype(float))
        dists, _ = tree.query(s2d[[x2, y2]].values.astype(float), k=min(3, len(s1d)))
        result = pd.DataFrame({'node_id': s2d['node_id'].values,
                               'dist_nearest_drain': dists[:, 0] if dists.ndim > 1 else dists})
        if dists.ndim > 1 and dists.shape[1] >= 2: result['dist_2nd_drain'] = dists[:, 1]
        if dists.ndim > 1 and dists.shape[1] >= 3: result['dist_3rd_drain'] = dists[:, 2]
        result['node_id'] = result['node_id'].astype('int64')
        print(f"  dist_nearest_drain Model_{model_id}: {len(result)} nodes")
        DIST_DRAIN_CACHE[model_id] = result
        return result
    except Exception as e:
        print(f"  [WARN] dist_to_drain failed: {e}")
        DIST_DRAIN_CACHE[model_id] = pd.DataFrame()
        return DIST_DRAIN_CACHE[model_id]

print("Dist-to-drain ready.")

Dist-to-drain ready.


In [8]:
# =========================================================

# RAIN FEATURES (v6: +future rain, +duration, +dry spell, +quantiles)

# =========================================================

RAIN_CACHE = {}

TIMESTEP_DURATION_CACHE = {}


def load_timestep_durations(model_id, event_dir):

    """v5.5: Load actual timestep durations from timesteps.csv."""

    k = (model_id, eid(event_dir))

    if k in TIMESTEP_DURATION_CACHE: return TIMESTEP_DURATION_CACHE[k]

    ts_path = event_dir / 'timesteps.csv'

    if not ts_path.exists():

        TIMESTEP_DURATION_CACHE[k] = None

        return None

    try:

        ts_df = read_csv(ts_path)

        # Expect columns like: timestep, datetime or time or seconds

        ts_col = fcol(ts_df.columns, ('timestep',), ('time','step'), False, 'ts')

        time_col = fcol(ts_df.columns, ('datetime','time','seconds','elapsed'),

                        ('time',), False, 'time')

        if ts_col is None or time_col is None:

            TIMESTEP_DURATION_CACHE[k] = None

            return None

        ts_df = ts_df.sort_values(ts_col).reset_index(drop=True)

        # Try to parse as seconds or datetime

        try:

            time_vals = pd.to_numeric(ts_df[time_col])

            durations = time_vals.diff().fillna(time_vals.iloc[0] if len(time_vals) > 0 else 0)

        except (ValueError, TypeError):

            try:

                time_vals = pd.to_datetime(ts_df[time_col])

                durations = time_vals.diff().dt.total_seconds().fillna(0)

            except:

                TIMESTEP_DURATION_CACHE[k] = None

                return None

        result = pd.DataFrame({

            'timestep': ts_df[ts_col].astype('int64'),

            'step_duration': durations.astype('float32'),

            'elapsed_time': time_vals.astype('float64') if pd.api.types.is_numeric_dtype(time_vals) else durations.cumsum().astype('float32'),

        })

        TIMESTEP_DURATION_CACHE[k] = result

        return result

    except Exception:

        TIMESTEP_DURATION_CACHE[k] = None

        return None


def load_rain(model_id, event_dir):

    k = (model_id, eid(event_dir))

    if k in RAIN_CACHE: return RAIN_CACHE[k]

    path_2d = dyn_path(event_dir, 2)

    df = read_csv(path_2d)

    df = ensure_nid(df)

    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')

    rain_col = fcol(df.columns, ('rainfall',), ('rain',), label='rainfall')

    rain_ts = df.groupby(ts_col)[rain_col].mean().reset_index()

    rain_ts.columns = ['timestep', 'rain_global']

    rain_ts = rain_ts.sort_values('timestep').reset_index(drop=True)

    rain = rain_ts['rain_global'].astype('float64')


    out = pd.DataFrame({

        'timestep': rain_ts['timestep'].astype('int64'),

        'rain_global': rain,

        'rain_lag1': rain.shift(1).fillna(0),

        'rain_lag2': rain.shift(2).fillna(0),

        'rain_lag3': rain.shift(3).fillna(0),

        'rain_roll3': rain.rolling(3, min_periods=1).sum(),

        'rain_roll6': rain.rolling(6, min_periods=1).sum(),

        'rain_roll12': rain.rolling(12, min_periods=1).sum(),

        'rain_roll24': rain.rolling(24, min_periods=1).sum(),

        'rain_cumsum': rain.cumsum(),

        'rain_delta': rain - rain.shift(1).fillna(0),

        'is_raining': (rain > 0).astype('int8'),

        'rain_intensity_peak': rain.expanding().max(),

    })


    out['rain_accel'] = out['rain_delta'] - out['rain_delta'].shift(1).fillna(0)

    for alpha in [0.1, 0.05, 0.02, 0.01]:

        out[f'rain_ema_{int(1/alpha)}'] = rain.ewm(alpha=alpha, adjust=False).mean()


    total = rain.sum()

    out['rain_cum_ratio'] = (rain.cumsum() / total) if total > 0 else 0


    peak = rain.max()

    peak_ts = rain_ts.loc[rain.idxmax(), 'timestep'] if peak > 0 else 0

    max_ts = rain_ts['timestep'].max()

    out['event_total_rain'] = total

    out['event_peak_rain'] = peak

    out['time_since_rain_peak'] = out['timestep'] - peak_ts

    out['frac_event'] = out['timestep'] / (max_ts + 1)

    out['rain_phase_rising'] = (out['rain_delta'] > 0).astype('int8')

    out['rain_phase_falling'] = ((out['rain_delta'] < 0) & (rain > 0)).astype('int8')


    # v5.5: Rain duration & dry spell

    is_rain = (rain > 0).values

    rain_dur = np.zeros(len(rain), dtype='int32')

    dry_spell = np.zeros(len(rain), dtype='int32')

    cnt_rain, cnt_dry = 0, 0

    for i in range(len(rain)):

        if is_rain[i]:

            cnt_rain += 1

            cnt_dry = 0

        else:

            cnt_dry += 1

            cnt_rain = 0

        rain_dur[i] = cnt_rain

        dry_spell[i] = cnt_dry

    out['rain_duration'] = rain_dur

    out['dry_spell'] = dry_spell


    # v5.5: Intensity quantiles (expanding)

    out['rain_q75'] = rain.expanding().quantile(0.75)

    out['rain_q90'] = rain.expanding().quantile(0.90)


    # v5.5: Rain rate of change (smoothed)

    out['rain_roc_smooth'] = rain.rolling(3, min_periods=1).mean().diff().fillna(0)


    # Extra rain-shape features guided by feature importance

    eps = 1e-6

    out['rain_recent_share_6'] = out['rain_roll6'] / (out['rain_cumsum'] + eps)

    out['rain_recent_share_12'] = out['rain_roll12'] / (out['rain_cumsum'] + eps)

    out['rain_recent_share_24'] = out['rain_roll24'] / (out['rain_cumsum'] + eps)

    out['rain_roll3_over_12'] = out['rain_roll3'] / (out['rain_roll12'] + eps)

    out['rain_roll6_over_24'] = out['rain_roll6'] / (out['rain_roll24'] + eps)

    out['rain_ema_ratio_10_100'] = (out['rain_ema_10'] + eps) / (out['rain_ema_100'] + eps)

    out['rain_ema_ratio_20_100'] = (out['rain_ema_20'] + eps) / (out['rain_ema_100'] + eps)

    out['rain_ema_ratio_50_100'] = (out['rain_ema_50'] + eps) / (out['rain_ema_100'] + eps)

    out['rain_q75_gap'] = rain - out['rain_q75']

    out['rain_q90_gap'] = rain - out['rain_q90']

    out['rain_peak_gap'] = out['rain_intensity_peak'] - rain

    out['rain_peak_gap_rel'] = out['rain_peak_gap'] / (out['rain_intensity_peak'] + eps)

    out['rain_global_over_q75'] = rain / (out['rain_q75'] + eps)

    out['rain_global_over_q90'] = rain / (out['rain_q90'] + eps)

    out['rain_cumsum_per_step'] = out['rain_cumsum'] / (out['timestep'] + 1.0)

    out['dry_spell_log'] = np.log1p(out['dry_spell'])

    out['rain_duration_log'] = np.log1p(out['rain_duration'])


    # v6: Future rain features (forecast fully available — organizer confirmed)

    rain_vals = rain.values

    n_rain = len(rain_vals)

    for h in [1, 3, 5, 10]:

        shifted = rain.shift(-h).fillna(0)

        out[f'rain_future_{h}'] = shifted.astype('float32')


    # Future rolling sums: sum of rain in next w steps (excluding current)

    for w in [5, 10]:

        cs = rain.cumsum().values

        total_cs = cs[-1]

        fsum = np.zeros(n_rain, dtype='float32')

        for i in range(n_rain):

            end_idx = min(i + w, n_rain - 1)

            fsum[i] = cs[end_idx] - cs[i]

        out[f'rain_future_sum{w}'] = fsum


    # Future max: max rain in next w steps

    for w in [5, 10]:

        fmax = np.zeros(n_rain, dtype='float32')

        for i in range(n_rain):

            end_idx = min(i + w + 1, n_rain)

            if i + 1 < end_idx:

                fmax[i] = rain_vals[i+1:end_idx].max()

        out[f'rain_future_max{w}'] = fmax


    # Rain remaining after current timestep

    out['rain_remaining'] = (total - rain.cumsum()).astype('float32')


    # Rain future acceleration (will it intensify or weaken?)

    rain_future_1 = rain.shift(-1).fillna(0)

    rain_future_3 = rain.shift(-3).fillna(0)

    out['rain_future_trend'] = (rain_future_3 - rain).astype('float32')



    # v8: Correlation-weighted rain cumsum (impulse response kernel)

    # From empirical analysis: corr(rain_lag_x, WL) has:

    #   - Peak ~0.25 at lag 30-50

    #   - Second bump ~0.20 at lag ~100

    #   - Long tail declining to ~0.05 at lag 200, ~0 by lag 300

    # We approximate this with a mixture of Gaussians + exponential tail

    max_lag = 200

    lags = np.arange(max_lag)


    # Approximate the empirical correlation curve

    # Peak 1: lag 40, sigma 20 (main hump)

    # Peak 2: lag 100, sigma 30 (secondary plateau)

    # Exponential tail

    corr_curve = (0.25 * np.exp(-0.5 * ((lags - 40) / 20) ** 2) +

                  0.15 * np.exp(-0.5 * ((lags - 100) / 30) ** 2) +

                  0.05 * np.exp(-lags / 150))

    corr_curve[:5] = np.linspace(0.12, corr_curve[5], 5)  # ramp up from lag 0

    corr_curve = np.maximum(corr_curve, 0)


    # Convolution using numpy (vectorized, fast)

    rain_vals_arr = rain.values.astype('float64')

    n_ts = len(rain_vals_arr)

    rain_padded = np.concatenate([np.zeros(max_lag), rain_vals_arr])


    # Main weighted cumsum (full impulse response)

    corr_weights = corr_curve / corr_curve.sum()

    # Use np.convolve for speed

    full_conv = np.convolve(rain_vals_arr, corr_weights[::-1], mode='full')[:n_ts]

    # Shift to align: convolve gives output where index t = sum of rain[t-k]*w[k]

    # We need causal: only past rain. np.convolve with 'full' and trim does this.

    # Actually let's do it properly with a reversed kernel

    out['rain_weighted_cumsum'] = np.convolve(

        rain_padded, corr_weights, mode='full'

    )[max_lag:max_lag + n_ts].astype('float32')


    # Short-term impulse response (fast drainage response, lag 10-60)

    w_short = np.exp(-0.5 * ((lags - 30) / 15) ** 2)

    w_short = w_short / w_short.sum()

    out['rain_impulse_short'] = np.convolve(

        rain_padded, w_short, mode='full'

    )[max_lag:max_lag + n_ts].astype('float32')


    # Medium-term impulse response (slow drainage, lag 50-150)

    w_medium = np.exp(-0.5 * ((lags - 90) / 30) ** 2)

    w_medium = w_medium / w_medium.sum()

    out['rain_impulse_medium'] = np.convolve(

        rain_padded, w_medium, mode='full'

    )[max_lag:max_lag + n_ts].astype('float32')


    # Long-term impulse response (tail, lag 100-200)

    w_long = np.exp(-0.5 * ((lags - 150) / 40) ** 2)

    w_long = w_long / w_long.sum()

    out['rain_impulse_long'] = np.convolve(

        rain_padded, w_long, mode='full'

    )[max_lag:max_lag + n_ts].astype('float32')


    # Extended rain lags (covering the full correlation curve)

    for lag in [5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150]:

        out[f'rain_lag{lag}'] = rain.shift(lag).fillna(0).astype('float32')


    # Extended rolling windows

    for w in [15, 20, 30, 50, 80, 100, 150]:

        out[f'rain_roll{w}'] = rain.rolling(w, min_periods=1).sum().astype('float32')


    # Ratio: recent vs total impulse (shows timing within flood event)

    eps = 1e-6

    out['rain_impulse_short_over_total'] = (

        out['rain_impulse_short'] / (out['rain_weighted_cumsum'] + eps)

    ).astype('float32')


    # v5.5: Timestep durations

    ts_dur = load_timestep_durations(model_id, event_dir)

    if ts_dur is not None:

        out = out.merge(ts_dur, on='timestep', how='left')

        if 'elapsed_time' in out.columns:

            out['rain_cumsum_per_time'] = out['rain_cumsum'] / (out['elapsed_time'].astype('float64') + 1.0)

            out['rain_roll6_per_time'] = out['rain_roll6'] / (out['elapsed_time'].astype('float64') + 1.0)


    RAIN_CACHE[k] = out

    return out


print("Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).")

Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).


In [9]:
# =========================================================
# WARMUP FEATURES + TRENDS + NEIGHBOR WARMUP
# + v5 NEW: INLET FLOW, WATER VOLUME, EDGE DYNAMICS
# =========================================================
WARMUP_CACHE = {}
WARMUP_DYN_CACHE = {}

def load_warmup(model_id, split, event_dir, node_type):
    key = (model_id, eid(event_dir), node_type, split)
    if key in WARMUP_CACHE: return WARMUP_CACHE[key]
    df = read_csv(dyn_path(event_dir, node_type))
    df = ensure_nid(df)
    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')
    wl_col = fcol(df.columns, ('water_level',), ('water','level'), label='water_level')
    warm = df[['node_id', ts_col, wl_col]].copy()
    warm.columns = ['node_id', 'timestep', 'water_level']
    warm['timestep'] = warm['timestep'].astype('int64')
    warm['water_level'] = warm['water_level'].astype('float64')
    warm = warm.sort_values(['node_id', 'timestep']).reset_index(drop=True)
    warm = warm[warm['timestep'] < WARMUP_STEPS].copy()
    warm['wi'] = warm.groupby('node_id').cumcount()
    wide = warm.pivot(index='node_id', columns='wi', values='water_level')
    wide.columns = [f'water_level_{int(c)}' for c in wide.columns]
    wide = wide.reset_index()
    for i in range(WARMUP_STEPS):
        c = f'water_level_{i}'
        if c not in wide.columns: wide[c] = np.nan
    WARMUP_CACHE[key] = wide
    return wide

def _agg_warmup_series(vals):
    clean = vals[~np.isnan(vals)]
    if len(clean) == 0:
        return np.nan, np.nan, np.nan, np.nan
    return np.mean(clean), np.max(clean), np.min(clean), clean[-1]

def load_warmup_dynamics(model_id, split, event_dir, node_type):
    key = (model_id, eid(event_dir), node_type, split)
    if key in WARMUP_DYN_CACHE: return WARMUP_DYN_CACHE[key]

    result_recs = {}

    # --- Node-level dynamic features (inlet_flow / water_volume) ---
    node_dyn = read_csv(dyn_path(event_dir, node_type))
    node_dyn = ensure_nid(node_dyn)
    ts_col = fcol(node_dyn.columns, ('timestep',), ('time','step'), label='timestep')
    node_dyn['timestep'] = node_dyn[ts_col].astype('int64')
    warmup_nodes = node_dyn[node_dyn['timestep'] < WARMUP_STEPS].copy()

    if node_type == 1:
        inlet_col = fcol(warmup_nodes.columns, ('1d_node_inlet_flow','inlet_flow'),
                         ('inlet','flow'), False, 'inlet_flow')
        if inlet_col:
            for nid, grp in warmup_nodes.groupby('node_id'):
                vals = grp[inlet_col].astype('float64').values
                mn, mx, mi, last = _agg_warmup_series(vals)
                result_recs.setdefault(int(nid), {})['inlet_flow_mean'] = mn
                result_recs.setdefault(int(nid), {})['inlet_flow_max'] = mx
                result_recs.setdefault(int(nid), {})['inlet_flow_min'] = mi
                result_recs.setdefault(int(nid), {})['inlet_flow_last'] = last
                abs_vals = np.abs(vals[~np.isnan(vals)])
                result_recs[int(nid)]['inlet_flow_abs_mean'] = np.mean(abs_vals) if len(abs_vals) else np.nan
                result_recs[int(nid)]['inlet_flow_abs_max'] = np.max(abs_vals) if len(abs_vals) else np.nan
    else:
        vol_col = fcol(warmup_nodes.columns, ('water_volume',), ('volume',), False, 'volume')
        if vol_col:
            for nid, grp in warmup_nodes.groupby('node_id'):
                vals = grp[vol_col].astype('float64').values
                mn, mx, mi, last = _agg_warmup_series(vals)
                result_recs.setdefault(int(nid), {})['wvol_mean'] = mn
                result_recs.setdefault(int(nid), {})['wvol_max'] = mx
                result_recs.setdefault(int(nid), {})['wvol_last'] = last
                clean = vals[~np.isnan(vals)]
                if len(clean) >= 2:
                    result_recs[int(nid)]['wvol_trend'] = clean[-1] - clean[0]
                else:
                    result_recs[int(nid)]['wvol_trend'] = np.nan

    del warmup_nodes, node_dyn

    # --- Edge-level dynamic features aggregated to nodes ---
    edge_path = edge_dyn_path(event_dir, node_type)
    k_ei = (model_id, node_type)
    if edge_path.exists() and k_ei in EDGE_INDEX_CACHE:
        fc_name, tc_name, ei_df = EDGE_INDEX_CACHE[k_ei]
        edge_dyn = read_csv(edge_path)
        ts_e = fcol(edge_dyn.columns, ('timestep',), ('time','step'), label='timestep')
        edge_dyn['timestep'] = edge_dyn[ts_e].astype('int64')
        warmup_edges = edge_dyn[edge_dyn['timestep'] < WARMUP_STEPS].copy()
        del edge_dyn

        if node_type == 1:
            flow_col = fcol(warmup_edges.columns, ('1d_edge_flow','edge_flow'),
                            ('edge','flow'), False, 'flow')
            vel_col = fcol(warmup_edges.columns, ('1d_edge_velocity','edge_velocity'),
                           ('velocity',), False, 'velocity')
        else:
            flow_col = fcol(warmup_edges.columns, ('2d_flow',), ('flow',), False, 'flow')
            vel_col = fcol(warmup_edges.columns, ('2d_velocity',), ('velocity',), False, 'velocity')

        if flow_col or vel_col:
            eidx_col = fcol(warmup_edges.columns, ('edge_idx','edge_id','link_id'),
                            ('edge',), False, 'edge_idx')

            edge_to_nodes = {}
            for i in range(len(ei_df)):
                edge_to_nodes[i] = (int(ei_df.iloc[i][fc_name]), int(ei_df.iloc[i][tc_name]))

            node_in_flows, node_out_flows = defaultdict(list), defaultdict(list)
            node_in_vels, node_out_vels = defaultdict(list), defaultdict(list)

            if eidx_col:
                edge_groups = warmup_edges.groupby(eidx_col)
            else:
                n_edges = len(ei_df)
                warmup_edges['_edge_idx'] = warmup_edges.groupby('timestep').cumcount()
                edge_groups = warmup_edges.groupby('_edge_idx')
                eidx_col = '_edge_idx'

            for edge_idx, grp in edge_groups:
                edge_idx = int(edge_idx)
                if edge_idx not in edge_to_nodes: continue
                from_n, to_n = edge_to_nodes[edge_idx]

                if flow_col and flow_col in grp.columns:
                    fvals = grp[flow_col].astype('float64').values
                    fmean = np.nanmean(fvals)
                    fabs_mean = np.nanmean(np.abs(fvals))
                    fabs_max = np.nanmax(np.abs(fvals))
                    node_out_flows[from_n].append((fmean, fabs_mean, fabs_max))
                    node_in_flows[to_n].append((fmean, fabs_mean, fabs_max))

                if vel_col and vel_col in grp.columns:
                    vvals = grp[vel_col].astype('float64').values
                    vmean = np.nanmean(vvals)
                    vabs_mean = np.nanmean(np.abs(vvals))
                    vabs_max = np.nanmax(np.abs(vvals))
                    node_out_vels[from_n].append((vmean, vabs_mean, vabs_max))
                    node_in_vels[to_n].append((vmean, vabs_mean, vabs_max))

            del warmup_edges

            prefix = '1d' if node_type == 1 else '2d'
            all_nids = set()
            for d in [node_in_flows, node_out_flows, node_in_vels, node_out_vels]:
                all_nids.update(d.keys())

            for nid in all_nids:
                rec = result_recs.setdefault(int(nid), {})

                if nid in node_in_flows:
                    in_f = node_in_flows[nid]
                    rec[f'{prefix}_in_flow_mean'] = np.mean([x[0] for x in in_f])
                    rec[f'{prefix}_in_flow_abs_mean'] = np.mean([x[1] for x in in_f])
                    rec[f'{prefix}_in_flow_abs_max'] = np.max([x[2] for x in in_f])

                if nid in node_out_flows:
                    out_f = node_out_flows[nid]
                    rec[f'{prefix}_out_flow_mean'] = np.mean([x[0] for x in out_f])
                    rec[f'{prefix}_out_flow_abs_mean'] = np.mean([x[1] for x in out_f])
                    rec[f'{prefix}_out_flow_abs_max'] = np.max([x[2] for x in out_f])

                if nid in node_in_vels:
                    in_v = node_in_vels[nid]
                    rec[f'{prefix}_in_vel_mean'] = np.mean([x[0] for x in in_v])
                    rec[f'{prefix}_in_vel_abs_mean'] = np.mean([x[1] for x in in_v])
                    rec[f'{prefix}_in_vel_abs_max'] = np.max([x[2] for x in in_v])

                if nid in node_out_vels:
                    out_v = node_out_vels[nid]
                    rec[f'{prefix}_out_vel_mean'] = np.mean([x[0] for x in out_v])
                    rec[f'{prefix}_out_vel_abs_mean'] = np.mean([x[1] for x in out_v])
                    rec[f'{prefix}_out_vel_abs_max'] = np.max([x[2] for x in out_v])

                if node_type == 1:
                    in_mean = rec.get(f'{prefix}_in_flow_mean', 0) or 0
                    out_mean = rec.get(f'{prefix}_out_flow_mean', 0) or 0
                    rec[f'{prefix}_net_flow'] = in_mean - out_mean

    if result_recs:
        rows = [{'node_id': nid, **feats} for nid, feats in result_recs.items()]
        result = pd.DataFrame(rows)
        result['node_id'] = result['node_id'].astype('int64')
    else:
        result = pd.DataFrame()

    WARMUP_DYN_CACHE[key] = result
    return result

def compute_warmup_trends(warmup_wide, static_df, node_type):
    wl_cols = sorted([c for c in warmup_wide.columns if c.startswith('water_level_')], key=lambda x: int(x.rsplit('_', 1)[1]))
    trend = pd.DataFrame({'node_id': warmup_wide['node_id'].astype('int64')})
    if not wl_cols:
        return trend

    vals = warmup_wide[wl_cols].values.astype('float64')
    trend['warmup_mean'] = np.nanmean(vals, axis=1)
    trend['warmup_std'] = np.nanstd(vals, axis=1)
    trend['warmup_range'] = np.nanmax(vals, axis=1) - np.nanmin(vals, axis=1)
    trend['warmup_last'] = vals[:, -1]

    recent3 = vals[:, -min(3, vals.shape[1]):]
    recent5 = vals[:, -min(5, vals.shape[1]):]
    trend['wl_recent3_mean'] = np.nanmean(recent3, axis=1)
    trend['wl_recent5_mean'] = np.nanmean(recent5, axis=1)
    trend['wl_recent3_std'] = np.nanstd(recent3, axis=1)
    trend['wl_recent5_std'] = np.nanstd(recent5, axis=1)
    trend['wl_last_vs_recent3'] = trend['warmup_last'] - trend['wl_recent3_mean']
    trend['wl_last_vs_recent5'] = trend['warmup_last'] - trend['wl_recent5_mean']
    trend['wl_peak_to_last'] = np.nanmax(vals, axis=1) - trend['warmup_last']
    trend['wl_last_to_min'] = trend['warmup_last'] - np.nanmin(vals, axis=1)

    if vals.shape[1] >= 2:
        trend['warmup_last_delta'] = vals[:, -1] - vals[:, -2]
        diffs = np.diff(vals, axis=1)
        trend['wl_diff_mean'] = np.nanmean(diffs, axis=1)
        trend['wl_diff_std'] = np.nanstd(diffs, axis=1)
        trend['wl_last3_diff_mean'] = np.nanmean(diffs[:, -min(3, diffs.shape[1]):], axis=1)
    if vals.shape[1] >= 3:
        trend['warmup_accel'] = (vals[:, -1] - vals[:, -2]) - (vals[:, -2] - vals[:, -3])
        trend['wl_recent3_slope'] = (vals[:, -1] - vals[:, -3]) / 2.0
    if vals.shape[1] >= 5:
        trend['wl_recent5_slope'] = (vals[:, -1] - vals[:, -5]) / 4.0

    x = np.arange(vals.shape[1], dtype='float64')
    xm = x.mean(); xvar = ((x - xm)**2).sum()
    if xvar > 0:
        trend['warmup_trend'] = np.nansum((x[None, :] - xm) * vals, axis=1) / xvar

    valid_mask = ~np.isnan(vals)
    vals_for_max = np.where(valid_mask, vals, -np.inf)
    vals_for_min = np.where(valid_mask, vals, np.inf)
    has_valid = valid_mask.any(axis=1)
    peak_idx = np.argmax(vals_for_max, axis=1).astype('float64')
    min_idx = np.argmin(vals_for_min, axis=1).astype('float64')
    peak_idx[~has_valid] = np.nan
    min_idx[~has_valid] = np.nan
    trend['warmup_peak_idx'] = peak_idx
    trend['warmup_min_idx'] = min_idx

    if node_type == 1:
        merge_cols = ['node_id']
        inv_c = fcol(static_df.columns, ('invert_elevation',), ('invert',), False, 'inv')
        surf_c = fcol(static_df.columns, ('surface_elevation',), ('surface',), False, 'surf')
        depth_c = fcol(static_df.columns, ('depth',), ('depth',), False, 'depth')
        area_c = fcol(static_df.columns, ('base_area',), ('base','area'), False, 'base_area')
        for c in [inv_c, surf_c, depth_c, area_c]:
            if c and c not in merge_cols:
                merge_cols.append(c)
        merged = trend.merge(static_df[merge_cols], on='node_id', how='left')

        inv_v = merged[inv_c].fillna(0).values if inv_c else np.zeros(len(merged))
        surf_v = merged[surf_c].fillna(0).values if surf_c else np.zeros(len(merged))
        pipe_d = surf_v - inv_v
        trend['fill_ratio'] = np.where(pipe_d > 0, (trend['warmup_last'].values - inv_v) / pipe_d, 0)
        trend['fill_ratio_clip'] = np.clip(trend['fill_ratio'], -2.0, 3.0)
        trend['is_surcharged'] = (trend['warmup_last'].values > surf_v).astype('int8') if surf_c else 0
        trend['head_above_invert'] = trend['warmup_last'].values - inv_v if inv_c else np.nan
        trend['headroom_to_surface'] = surf_v - trend['warmup_last'].values if surf_c else np.nan
        trend['recent3_head_above_invert'] = trend['wl_recent3_mean'].values - inv_v if inv_c else np.nan
        trend['recent3_headroom_to_surface'] = surf_v - trend['wl_recent3_mean'].values if surf_c else np.nan
        trend['warmup_peak_to_surface_gap'] = surf_v - np.nanmax(vals, axis=1) if surf_c else np.nan
        if depth_c:
            depth_v = merged[depth_c].fillna(0).values
            trend['head_above_invert_over_depth'] = np.where(depth_v > 0, trend['head_above_invert'] / depth_v, 0)
            trend['headroom_to_surface_over_depth'] = np.where(depth_v > 0, trend['headroom_to_surface'] / depth_v, 0)
            trend['depth_x_fill'] = depth_v * trend['fill_ratio']
        if area_c:
            area_v = merged[area_c].fillna(0).values
            trend['base_area_x_head'] = area_v * np.clip(trend['head_above_invert'], 0, None)
            trend['base_area_x_fill'] = area_v * trend['fill_ratio']
    return trend

def compute_neighbor_warmup(model_id, node_type, warmup_wide):
    adj = ADJ_CACHE.get((model_id, node_type), {})
    if not adj or 'water_level_9' not in warmup_wide.columns:
        return pd.DataFrame()
    wl9_map = dict(zip(warmup_wide['node_id'].astype(int),
                       warmup_wide['water_level_9'].astype('float64')))
    records = []
    for nid in warmup_wide['node_id'].astype(int):
        nid = int(nid)
        nbrs = adj.get(nid, [])
        own_wl9 = wl9_map.get(nid, np.nan)
        if nbrs:
            nbr_wl9 = [wl9_map.get(n, np.nan) for n in nbrs]
            nbr_mean = np.nanmean(nbr_wl9)
            records.append({'node_id': nid, 'nbr_mean_wl9': nbr_mean,
                           'nbr_max_wl9': np.nanmax(nbr_wl9),
                           'gradient_wl9': (own_wl9 - nbr_mean) if pd.notna(own_wl9) and pd.notna(nbr_mean) else np.nan})
        else:
            records.append({'node_id': nid, 'nbr_mean_wl9': np.nan,
                           'nbr_max_wl9': np.nan, 'gradient_wl9': np.nan})
    return pd.DataFrame(records)

print("Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).")

Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).


In [10]:
# =========================================================

# BUILD SUPERVISED FRAME (v6: +spectral embeddings, +aux targets)

# =========================================================

EVENT_FRAME_CACHE = {}


def add_importance_guided_features(df, node_type):

    eps = 1e-6


    x_col = fcol(df.columns, ('position_x',), ('position', 'x'), False, 'position_x')

    y_col = fcol(df.columns, ('position_y',), ('position', 'y'), False, 'position_y')

    if x_col and y_col:

        x = df[x_col].astype('float64')

        y = df[y_col].astype('float64')

        df['pos_radius'] = np.sqrt(x * x + y * y)

        df['pos_manhattan'] = np.abs(x) + np.abs(y)

        df['pos_angle'] = np.arctan2(y, x)

        df['pos_xy_sum'] = x + y

        df['pos_xy_diff'] = x - y


    if 'fill_ratio' in df.columns and 'rain_ema_50' in df.columns:

        df['fill_x_rain_ema_50'] = df['fill_ratio'] * df['rain_ema_50']

    if 'fill_ratio' in df.columns and 'rain_ema_100' in df.columns:

        df['fill_x_rain_ema_100'] = df['fill_ratio'] * df['rain_ema_100']

    if 'headroom_to_surface' in df.columns and 'rain_cumsum' in df.columns:

        df['headroom_x_rain_cumsum'] = df['headroom_to_surface'] * df['rain_cumsum']

    if 'head_above_invert' in df.columns and 'rain_ema_20' in df.columns:

        df['head_x_rain_ema_20'] = df['head_above_invert'] * df['rain_ema_20']

    if 'wl_recent3_mean' in df.columns and 'rain_cumsum' in df.columns:

        df['wl_recent3_x_rain_cumsum'] = df['wl_recent3_mean'] * df['rain_cumsum']

    if 'warmup_last' in df.columns and 'nbr_mean_wl9' in df.columns:

        df['warmup_last_minus_nbr_mean'] = df['warmup_last'] - df['nbr_mean_wl9']

    if 'warmup_last' in df.columns and 'nbr_max_wl9' in df.columns:

        df['warmup_last_minus_nbr_max'] = df['warmup_last'] - df['nbr_max_wl9']

    if 'water_level_9' in df.columns and 'hop2_elev_min' in df.columns:

        df['wl9_minus_hop2_elev_min'] = df['water_level_9'] - df['hop2_elev_min']

    if 'water_level_9' in df.columns and 'hop2_elev_mean' in df.columns:

        df['wl9_minus_hop2_elev_mean'] = df['water_level_9'] - df['hop2_elev_mean']

    if 'pipe_cap_total_out' in df.columns and 'pipe_cap_total_in' in df.columns:

        df['pipe_cap_net_out'] = df['pipe_cap_total_out'] - df['pipe_cap_total_in']

        df['pipe_cap_ratio_out_in'] = df['pipe_cap_total_out'] / (df['pipe_cap_total_in'] + eps)

    if 'upstream_total' in df.columns and 'pipe_cap_total_out' in df.columns:

        df['upstream_per_outcap'] = df['upstream_total'] / (df['pipe_cap_total_out'] + eps)

    if 'inlet_flow_abs_mean' in df.columns and 'pipe_cap_total_in' in df.columns:

        df['inlet_to_cap_ratio'] = df['inlet_flow_abs_mean'] / (df['pipe_cap_total_in'] + eps)

    if 'coupling_elev_diff' in df.columns and 'fill_ratio' in df.columns:

        df['coupling_x_fill'] = df['coupling_elev_diff'] * df['fill_ratio']


    return df


def build_supervised_event_frame(model_id, split, event_dir, node_type, include_target=True):

    cache_key = (model_id, eid(event_dir), node_type, split, include_target)

    if cache_key in EVENT_FRAME_CACHE:

        return EVENT_FRAME_CACHE[cache_key].copy()


    df_dyn = read_csv(dyn_path(event_dir, node_type))

    df_dyn = ensure_nid(df_dyn)

    ts_col = fcol(df_dyn.columns, ('timestep',), ('time','step'), label='timestep')

    wl_col = fcol(df_dyn.columns, ('water_level',), ('water','level'),

                  required=include_target, label='water_level')

    rain_col = fcol(df_dyn.columns, ('rainfall',), ('rain',), required=False, label='rainfall')


    keep = [ts_col, 'node_id']

    if include_target and wl_col: keep.append(wl_col)

    if node_type == 2 and rain_col: keep.append(rain_col)


    df = df_dyn[keep].copy()

    del df_dyn

    renames = {ts_col: 'timestep'}

    if include_target and wl_col: renames[wl_col] = 'water_level'

    if node_type == 2 and rain_col: renames[rain_col] = 'rain_local'

    df = df.rename(columns=renames)

    df['timestep'] = df['timestep'].astype('int64')

    df = df[df['timestep'] >= MIN_PRED_TIMESTEP].copy()


    df['model_id'] = model_id

    df['event_id'] = eid(event_dir)

    df['node_type'] = node_type


    # Static features

    static = load_static(model_id, node_type)

    df = df.merge(static, on='node_id', how='left')


    # Rain features

    rain = load_rain(model_id, event_dir)

    df = df.merge(rain, on='timestep', how='left')


    # Warmup water levels

    warmup = load_warmup(model_id, split, event_dir, node_type)

    df = df.merge(warmup, on='node_id', how='left')


    # Graph features

    gf = load_graph_feats(model_id, node_type)

    if len(gf) > 0 and 'node_id' in gf.columns:

        df = df.merge(gf, on='node_id', how='left')


    # Distance to drain (2D)

    if node_type == 2:

        dd = load_dist_to_drain(model_id)

        if len(dd) > 0 and 'node_id' in dd.columns:

            df = df.merge(dd, on='node_id', how='left')


    # Warmup trends

    trend = compute_warmup_trends(warmup, static, node_type)

    df = df.merge(trend, on='node_id', how='left')


    # Neighbor warmup

    nbr = compute_neighbor_warmup(model_id, node_type, warmup)

    if len(nbr) > 0:

        df = df.merge(nbr, on='node_id', how='left')


    # Warmup dynamics (inlet_flow, water_volume, edge flow/velocity)

    wdyn = load_warmup_dynamics(model_id, split, event_dir, node_type)

    if len(wdyn) > 0 and 'node_id' in wdyn.columns:

        df = df.merge(wdyn, on='node_id', how='left')


    if 'rain_local' not in df.columns:

        df['rain_local'] = df.get('rain_global', 0)


    # Extra handcrafted features guided by current importances

    df = add_importance_guided_features(df, node_type)


    # Interaction features (1D only)

    if 'upstream_total' in df.columns:

        df['upstream_x_rain_cumsum'] = df['upstream_total'] * df['rain_cumsum']

        df['upstream_x_rain_ema_50'] = df['upstream_total'] * df.get('rain_ema_50', 0)


    if 'inlet_flow_abs_mean' in df.columns:

        df['inlet_x_rain_cumsum'] = df['inlet_flow_abs_mean'] * df['rain_cumsum']


    if 'fill_ratio' in df.columns:

        df['fill_x_rain_cumsum'] = df['fill_ratio'] * df['rain_cumsum']


    if 'pipe_cap_total_in' in df.columns:

        df['cap_x_rain_cumsum'] = df['pipe_cap_total_in'] * df['rain_cumsum']

        df['cap_x_fill'] = df['pipe_cap_total_in'] * df.get('fill_ratio', 0)


    # v6: Spectral graph embeddings

    se = compute_spectral_embeddings(model_id, node_type)

    if len(se) > 0 and 'node_id' in se.columns:

        df = df.merge(se, on='node_id', how='left')


    # v6: Auxiliary target predictions (inlet_flow)

    if USE_AUX_TARGETS and 'AUX_MODELS' in dir() and (model_id, node_type) in AUX_MODELS:

        aux_lgb, feat_aux = AUX_MODELS[(model_id, node_type)]

        avail = [c for c in feat_aux if c in df.columns]

        if len(avail) == len(feat_aux):

            df['pred_inlet_flow'] = aux_lgb.predict(df[feat_aux]).astype('float32')

            df['pred_inlet_x_rain'] = df['pred_inlet_flow'] * df.get('rain_cumsum', 0)

        else:

            df['pred_inlet_flow'] = np.float32(0)

            df['pred_inlet_x_rain'] = np.float32(0)

    elif USE_AUX_TARGETS:

        df['pred_inlet_flow'] = np.float32(0)

        df['pred_inlet_x_rain'] = np.float32(0)



    # v8: Node scale factor for ratio target

    # WL(node, t) ≈ f(t) * g(node) — multiplicative structure

    # g(node) = node_scale = std of water_level across timesteps during warmup

    # This allows predicting ratio = WL / node_scale instead of raw delta

    if include_target and 'water_level_9' in df.columns:

        # node_scale from warmup: range of water levels during warmup period

        warmup_cols_here = [c for c in df.columns if re.fullmatch(r'water_level_\d+', c)]

        if warmup_cols_here:

            warmup_vals = df[warmup_cols_here].values.astype('float64')

            node_range = warmup_vals.max(axis=1) - warmup_vals.min(axis=1)

            node_mean = warmup_vals.mean(axis=1)

            # Use warmup mean as scale (avoid division by zero)

            df['node_scale'] = np.where(

                np.abs(node_mean) > 0.01,

                np.abs(node_mean),

                1.0

            ).astype('float32')

            df['node_warmup_range'] = node_range.astype('float32')

    elif 'water_level_9' in df.columns:

        warmup_cols_here = [c for c in df.columns if re.fullmatch(r'water_level_\d+', c)]

        if warmup_cols_here:

            warmup_vals = df[warmup_cols_here].values.astype('float64')

            node_mean = warmup_vals.mean(axis=1)

            df['node_scale'] = np.where(

                np.abs(node_mean) > 0.01,

                np.abs(node_mean),

                1.0

            ).astype('float32')

            df['node_warmup_range'] = (warmup_vals.max(axis=1) - warmup_vals.min(axis=1)).astype('float32')


    df['node_id_cat'] = df['node_id'].astype(str)

    df['log_timestep'] = np.log1p(df['timestep'].astype('float32'))


    if include_target and USE_TARGET_DELTA and 'water_level_9' in df.columns:

        df['wl_delta_target'] = df['water_level'].astype('float64') - df['water_level_9'].astype('float64')


    for c in df.columns:

        if c == 'node_id_cat': continue

        if df[c].dtype == 'object':

            conv = pd.to_numeric(df[c], errors='coerce')

            if conv.notna().mean() >= 0.95: df[c] = conv

            else: df[c] = df[c].fillna('NA').astype(str)


    df = df.sort_values(['node_id', 'timestep']).reset_index(drop=True)

    if include_target:

        df['water_level'] = df['water_level'].astype('float64')


    # Downcast to float32 (keep target as float64)

    for c in df.columns:

        if c in (TARGET_COL, 'wl_delta_target'): continue

        if df[c].dtype == 'float64':

            df[c] = df[c].astype('float32')


    EVENT_FRAME_CACHE[cache_key] = df.copy()

    return df


def get_features(df):

    excluded = set(KEY_COLS + [TARGET_COL, 'event_id', 'wl_delta_target'])

    feat = [c for c in df.columns if c not in excluded]

    cat = [c for c in feat if df[c].dtype == 'object' or c.endswith('_cat')]

    return feat, cat


print("Frame builder ready (v6: +spectral embeddings, +future rain).")

Frame builder ready (v6: +spectral embeddings, +future rain).


In [11]:
# =========================================================
# DATASET INDEXING & SPLIT
# =========================================================
assert DATA_ROOT.exists()
model_dirs = sorted(DATA_ROOT.glob('Model_*'))
assert len(model_dirs) == 2

train_events, test_events = {}, {}
for d in model_dirs:
    m = mid_fn(d)
    train_events[m] = event_dirs(d / 'train')
    test_events[m] = event_dirs(d / 'test')
    print(f"Model {m}: {len(train_events[m])} train, {len(test_events[m])} test")

train_pool, valid_pool = {}, {}
for m in sorted(train_events):
    tr, va = train_test_split(train_events[m], test_size=VALID_EVENTS_PER_MODEL,
                               random_state=RANDOM_SEED + m, shuffle=True)
    train_pool[m] = sorted(tr, key=eid)
    valid_pool[m] = sorted(va, key=eid)
    print(f"  Split: {len(train_pool[m])} train, {len(valid_pool[m])} valid")

Model 1: 68 train, 29 test
Model 2: 69 train, 30 test
  Split: 53 train, 15 valid
  Split: 54 train, 15 valid


In [12]:
# =========================================================

# BUILD DATAFRAMES ONLY FOR (model_id=2, node_type=1)

# =========================================================

TARGET_MODEL_ID = 2

TARGET_NODE_TYPE = 1

print(f"Preparing data for Model {TARGET_MODEL_ID}, NodeType {TARGET_NODE_TYPE}")


EVENT_FRAME_CACHE.clear()

RAIN_CACHE.clear()

WARMUP_CACHE.clear()

WARMUP_DYN_CACHE.clear()

_free_memory()


train_frames = [

    build_supervised_event_frame(TARGET_MODEL_ID, 'train', ed, TARGET_NODE_TYPE, include_target=True)

    for ed in train_pool[TARGET_MODEL_ID]

]

eval_frames = [

    build_supervised_event_frame(TARGET_MODEL_ID, 'train', ed, TARGET_NODE_TYPE, include_target=True)

    for ed in valid_pool[TARGET_MODEL_ID]

]


train_df = pd.concat(train_frames, ignore_index=True)

eval_df = pd.concat(eval_frames, ignore_index=True)

del train_frames, eval_frames


EVENT_FRAME_CACHE.clear()

RAIN_CACHE.clear()

WARMUP_CACHE.clear()

WARMUP_DYN_CACHE.clear()

_free_memory()


feature_cols, cat_cols = get_features(train_df)

# v8: Support ratio target

if USE_TARGET_RATIO and 'wl_ratio_target' in train_df.columns:

    target_col = 'wl_ratio_target'

elif USE_TARGET_DELTA and 'wl_delta_target' in train_df.columns:

    target_col = 'wl_delta_target'

else:

    target_col = TARGET_COL


ordered_cols = []

for c in KEY_COLS + [TARGET_COL, 'wl_delta_target'] + feature_cols:

    if c in train_df.columns and c not in ordered_cols:

        ordered_cols.append(c)


train_df = train_df[ordered_cols].copy()

eval_df = eval_df[[c for c in ordered_cols if c in eval_df.columns]].copy()


print(f"train_df: {train_df.shape}")

print(f"eval_df : {eval_df.shape}")


# --- Build test frame for inference ---

EVENT_FRAME_CACHE.clear()

RAIN_CACHE.clear()

WARMUP_CACHE.clear()

WARMUP_DYN_CACHE.clear()

_free_memory()


test_frames = [

    build_supervised_event_frame(TARGET_MODEL_ID, 'test', ed, TARGET_NODE_TYPE, include_target=False)

    for ed in test_events[TARGET_MODEL_ID]

]

test_df = pd.concat(test_frames, ignore_index=True)

del test_frames


EVENT_FRAME_CACHE.clear()

RAIN_CACHE.clear()

WARMUP_CACHE.clear()

WARMUP_DYN_CACHE.clear()

_free_memory()


test_cols = [c for c in KEY_COLS + feature_cols if c in test_df.columns]

test_df = test_df[test_cols].copy()


print(f"test_df : {test_df.shape}")

Preparing data for Model 2, NodeType 1
  Model 2 1d edge_index: 197 edges, cols=['edge_idx', 'from_node', 'to_node']
    Edge static: 197 edges, cols=['edge_idx', 'relative_position_x', 'relative_position_y', 'length', 'diameter', 'shape', 'roughness', 'slope']
    Upstream total: max=197
    Centrality: PageRank + degree for 198 nodes
    2-hop elev features: 198 nodes
    Pipe capacity: 198 nodes
    Graph features: 198 nodes, 81 features
train_df: (2545884, 270)
eval_df : (852390, 270)
test_df : (1422036, 268)


In [13]:
dataset_info = {
    'model_id': TARGET_MODEL_ID,
    'node_type': TARGET_NODE_TYPE,
    'target_col': target_col,
    'feature_cols': feature_cols,
    'cat_cols': cat_cols,
    'n_features': len(feature_cols),
    'train_rows': len(train_df),
    'eval_rows': len(eval_df),
    'test_rows': len(test_df),
}
dataset_info

{'model_id': 2,
 'node_type': 1,
 'target_col': 'wl_delta_target',
 'feature_cols': ['timestep',
  'position_x',
  'position_y',
  'depth',
  'invert_elevation',
  'surface_elevation',
  'base_area',
  'rain_global',
  'rain_lag1',
  'rain_lag2',
  'rain_lag3',
  'rain_roll3',
  'rain_roll6',
  'rain_roll12',
  'rain_roll24',
  'rain_cumsum',
  'rain_delta',
  'is_raining',
  'rain_intensity_peak',
  'rain_accel',
  'rain_ema_10',
  'rain_ema_20',
  'rain_ema_50',
  'rain_ema_100',
  'rain_cum_ratio',
  'event_total_rain',
  'event_peak_rain',
  'time_since_rain_peak',
  'frac_event',
  'rain_phase_rising',
  'rain_phase_falling',
  'rain_duration',
  'dry_spell',
  'rain_q75',
  'rain_q90',
  'rain_roc_smooth',
  'rain_recent_share_6',
  'rain_recent_share_12',
  'rain_recent_share_24',
  'rain_roll3_over_12',
  'rain_roll6_over_24',
  'rain_ema_ratio_10_100',
  'rain_ema_ratio_20_100',
  'rain_ema_ratio_50_100',
  'rain_q75_gap',
  'rain_q90_gap',
  'rain_peak_gap',
  'rain_peak_gap_

In [14]:
# =========================================================
# CONVENIENCE VIEWS
# =========================================================
train_X = train_df[dataset_info['feature_cols']].copy()
eval_X = eval_df[dataset_info['feature_cols']].copy()
test_X = test_df[dataset_info['feature_cols']].copy()

train_y = train_df[dataset_info['target_col']].copy() if dataset_info['target_col'] in train_df.columns else None
eval_y = eval_df[dataset_info['target_col']].copy() if dataset_info['target_col'] in eval_df.columns else None

print("train_X:", train_X.shape)
print("eval_X :", eval_X.shape)
print("test_X :", test_X.shape)
if train_y is not None:
    print("train_y:", train_y.shape)
if eval_y is not None:
    print("eval_y :", eval_y.shape)

train_X: (2545884, 264)
eval_X : (852390, 264)
test_X : (1422036, 264)
train_y: (2545884,)
eval_y : (852390,)


In [15]:
# =========================================================

# MODELS: FloodDLModel + Rain-Window Preparation

# =========================================================

import math

import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rtdl_num_embeddings'])


cb_valid_model = None

cb_valid_pred = None

cb_best_iter = None

cb_valid_rmse = None

cb_specs = {}

cb_refit_models = {}

cb_refit_meta = []


cb_test_pred_by_seed = {}


m = dataset_info['model_id']

nt = dataset_info['node_type']

feat = dataset_info['feature_cols']

cat = dataset_info['cat_cols']

target = dataset_info['target_col']

use_delta = (target == 'wl_delta_target')

sd = STD_DEV_DICT.get((m, nt), 1.0)


cb_params = dict(CB_PARAMS[(m, nt)])


print(f"\n{'=' * 60}")

print(f"=== Model {m}, NodeType {nt} — CatBoost + Multi-DL ensemble ===")

print(f"  Features: {len(feat)}, Rows: train={len(train_df):,} valid={len(eval_df):,} test={len(test_df):,}")

print(f"  Target: {target}")

print(f"  DL experiments: {list(DL_EXPERIMENTS.keys())}")



def seed_everything(seed: int) -> None:

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)



def _embedding_dim(cardinality: int) -> int:

    return int(min(64, max(4, round(1.6 * (cardinality ** 0.56)))))



# -----------------------------------------------------------------

# Rain-window data preparation

# -----------------------------------------------------------------

def _prepare_rain_window_inputs(train_df, eval_df=None, test_df=None,

                                 feature_cols=None, cat_cols=None,

                                 target_col='wl_delta_target',

                                 past_w=120, future_w=10):

    """Prepare inputs with per-timestep rain window sequences.


    For each row (node, timestep_t):

      - Extract rain series [t-past_w .. t+future_w] = past_w + future_w + 1 steps

      - Each step has 3 channels: [rain_global, rain_cumsum_at_step, is_raining]

      - Prepend 10 warmup WL values: [water_level_i, 0, 0]

      - Total seq_len = 10 + past_w + future_w + 1


    Removes rain lag/roll features from numeric (now in sequence).

    """

    feature_cols = list(feature_cols)


    # Warmup columns

    warmup_cols = sorted(

        [c for c in feature_cols if re.fullmatch(r'water_level_\d+', c)],

        key=lambda x: int(x.split('_')[-1])

    )

    n_warmup = len(warmup_cols)  # typically 10


    # Rain features to DROP from numeric (now captured in sequence)

    rain_seq_drop = {'rain_lag1', 'rain_lag2', 'rain_lag3',

                     'rain_roll3', 'rain_roll6', 'rain_delta',

                     'rain_accel', 'rain_roll12', 'rain_roll24',

                     'rain_global', 'is_raining'}


    cat_cols = [c for c in (cat_cols or []) if c in feature_cols]

    num_cols = [c for c in feature_cols

                if c not in set(warmup_cols) and c not in rain_seq_drop

                and c not in cat_cols]


    seq_len = n_warmup + past_w + future_w + 1

    seq_channels = 3  # rain_global, rain_cumsum, is_raining


    prep = {

        'warmup_cols': warmup_cols,

        'num_cols': num_cols,

        'cat_cols': cat_cols,

        'target_col': target_col,

        'seq_len': seq_len,

        'seq_channels': seq_channels,

        'n_warmup': n_warmup,

        'past_w': past_w,

        'future_w': future_w,

    }


    # Build rain series lookup: event_id -> {timestep -> (rain, cumsum, is_raining)}

    rain_lookup = {}

    for eid_val in set(train_df['event_id'].unique()):

        rain_key = (m, eid_val)

        if rain_key in RAIN_CACHE:

            rc = RAIN_CACHE[rain_key]

            rain_ts = rc['timestep'].values

            rain_vals = rc['rain_global'].values.astype('float32')

            rain_cs = rc['rain_cumsum'].values.astype('float32')

            rain_is = (rain_vals > 0).astype('float32')

            rain_lookup[eid_val] = {

                'timesteps': rain_ts,

                'rain': rain_vals,

                'cumsum': rain_cs,

                'is_raining': rain_is,

                'max_ts': int(rain_ts.max()) if len(rain_ts) > 0 else 0,

            }


    # Also add eval/test event rain

    for extra_df in [eval_df, test_df]:

        if extra_df is None:

            continue

        for eid_val in set(extra_df['event_id'].unique()):

            if eid_val in rain_lookup:

                continue

            rain_key = (m, eid_val)

            if rain_key in RAIN_CACHE:

                rc = RAIN_CACHE[rain_key]

                rain_ts = rc['timestep'].values

                rain_vals = rc['rain_global'].values.astype('float32')

                rain_cs = rc['rain_cumsum'].values.astype('float32')

                rain_is = (rain_vals > 0).astype('float32')

                rain_lookup[eid_val] = {

                    'timesteps': rain_ts,

                    'rain': rain_vals,

                    'cumsum': rain_cs,

                    'is_raining': rain_is,

                    'max_ts': int(rain_ts.max()) if len(rain_ts) > 0 else 0,

                }


    # Build indexed rain arrays per event for fast lookup

    # Pad with past_w zeros at start and future_w zeros at end for safe indexing

    rain_arrays = {}

    for eid_val, rd in rain_lookup.items():

        max_ts = rd['max_ts']

        arr_size = max_ts + 1 + past_w + future_w + 1

        # Offset: index 0 in padded array = timestep (-past_w)

        # So timestep t maps to padded index (t + past_w)

        rain_arr = np.zeros(arr_size, dtype='float32')

        cs_arr = np.zeros_like(rain_arr)

        is_arr = np.zeros_like(rain_arr)

        for i, ts in enumerate(rd['timesteps']):

            idx = int(ts) + past_w

            if 0 <= idx < arr_size:

                rain_arr[idx] = rd['rain'][i]

                cs_arr[idx] = rd['cumsum'][i]

                is_arr[idx] = rd['is_raining'][i]

        rain_arrays[eid_val] = (rain_arr, cs_arr, is_arr, arr_size)


    prep['rain_arrays'] = rain_arrays


    rain_window_len = past_w + future_w + 1  # 35 steps


    def build_sequences(df):

        """Build rain-window sequences for each row (vectorized per event)."""

        n = len(df)

        seq = np.zeros((n, seq_len, seq_channels), dtype='float32')


        # Fill warmup WL part (first n_warmup steps, channel 0 only)

        wl_vals = df[warmup_cols].values.astype('float32')

        seq[:, :n_warmup, 0] = wl_vals


        # Fill rain window part — vectorized per event

        event_ids = df['event_id'].values

        timesteps = df['timestep'].values.astype(np.int64)


        for eid_val, (r_arr, c_arr, is_arr, arr_size) in rain_arrays.items():

            mask = event_ids == eid_val

            if not mask.any():

                continue

            ts_vals = timesteps[mask]

            indices = np.where(mask)[0]


            # For each row, we need indices [ts-past_w .. ts+future_w] in the padded array

            # In padded array, timestep t maps to index (t + past_w)

            # So ts-past_w maps to index ts, and ts+future_w maps to index ts+past_w+future_w

            # Window start in padded array = ts (since offset = past_w)

            starts = ts_vals  # start index in padded array


            for j in range(rain_window_len):

                read_idx = starts + j

                # Clip to valid range

                valid = (read_idx >= 0) & (read_idx < arr_size)

                read_idx_safe = np.clip(read_idx, 0, arr_size - 1)

                seq_col = n_warmup + j


                vals_r = r_arr[read_idx_safe]

                vals_c = c_arr[read_idx_safe]

                vals_i = is_arr[read_idx_safe]


                vals_r[~valid] = 0

                vals_c[~valid] = 0

                vals_i[~valid] = 0


                seq[indices, seq_col, 0] = vals_r

                seq[indices, seq_col, 1] = vals_c

                seq[indices, seq_col, 2] = vals_i


        return seq


    # Scaler for numeric features

    scaler = None

    if num_cols:

        scaler = StandardScaler()

        train_num = scaler.fit_transform(train_df[num_cols].astype('float32'))

        train_num = np.asarray(train_num, dtype=np.float32)

    else:

        train_num = np.zeros((len(train_df), 0), dtype=np.float32)

    prep['scaler'] = scaler


    # Categorical encoding

    cat_cardinalities = []

    categories_by_col = {}

    for c in cat_cols:

        parts = [train_df[c].astype('string').fillna('__NA__')]

        if eval_df is not None:

            parts.append(eval_df[c].astype('string').fillna('__NA__'))

        if test_df is not None:

            parts.append(test_df[c].astype('string').fillna('__NA__'))

        categories = pd.Index(pd.concat(parts, ignore_index=True).unique())

        categories_by_col[c] = categories

        cat_cardinalities.append(int(len(categories)))

    prep['cat_cardinalities'] = cat_cardinalities


    def _build_split(df, num_array=None):

        if df is None:

            return None


        # Build rain-window sequences

        seq = build_sequences(df)


        if num_cols:

            if num_array is None:

                num_array = scaler.transform(df[num_cols].astype('float32'))

            num = np.asarray(num_array, dtype=np.float32)

        else:

            num = np.zeros((len(df), 0), dtype=np.float32)


        if cat_cols:

            cat_list = []

            for c in cat_cols:

                codes = pd.Categorical(

                    df[c].astype('string').fillna('__NA__'),

                    categories=categories_by_col[c]

                ).codes.astype(np.int64)

                cat_list.append(codes)

            cat_arr = np.column_stack(cat_list).astype(np.int64, copy=False)

        else:

            cat_arr = np.zeros((len(df), 0), dtype=np.int64)


        out = {

            'seq': seq,

            'num': num,

            'cat': cat_arr,

            'wl9': np.asarray(df['water_level_9'].astype('float32').to_numpy(copy=False), dtype=np.float32),

        }

        if target_col in df.columns:

            out['y'] = np.asarray(df[target_col].astype('float32').to_numpy(copy=False), dtype=np.float32)

        return out


    print(f"  Building rain-window sequences (past={past_w}, future={future_w}, "

          f"seq_len={seq_len}, channels={seq_channels})...")


    t0 = time.time()

    prep['train'] = _build_split(train_df, train_num)

    prep['eval'] = _build_split(eval_df)

    prep['test'] = _build_split(test_df)

    elapsed = time.time() - t0

    print(f"  Rain-window sequences built in {elapsed:.1f}s")

    print(f"  Train seq shape: {prep['train']['seq'].shape}")

    print(f"  Num cols: {len(num_cols)}, Cat cols: {len(cat_cols)}")


    # Verify: different timesteps for same node should have different sequences

    if len(train_df) > 100:

        node_groups = train_df.groupby('node_id')

        sample_node = list(node_groups.groups.keys())[0]

        node_mask = train_df['node_id'] == sample_node

        node_seqs = prep['train']['seq'][node_mask.values]

        if len(node_seqs) > 1:

            diffs = np.abs(node_seqs[0] - node_seqs[1]).sum()

            print(f"  Verification: seq diff between timesteps of node {sample_node}: {diffs:.4f} "

                  f"({'OK - different' if diffs > 0 else 'WARNING - identical!'})")


    return prep



# Also keep legacy _prepare_bigru_inputs for baseline comparison

def _prepare_bigru_inputs(train_df, eval_df=None, test_df=None, feature_cols=None, cat_cols=None, target_col='wl_delta_target'):

    feature_cols = list(feature_cols)

    seq_cols = sorted(

        [c for c in feature_cols if re.fullmatch(r'water_level_\d+', c)],

        key=lambda x: int(x.split('_')[-1])

    )

    if not seq_cols:

        raise ValueError("BiGRU expects warmup sequence columns like water_level_0..9")


    cat_cols = [c for c in (cat_cols or []) if c in feature_cols]

    num_cols = [c for c in feature_cols if c not in set(seq_cols + cat_cols)]


    prep = {

        'seq_cols': seq_cols,

        'cat_cols': cat_cols,

        'num_cols': num_cols,

        'target_col': target_col,

        'seq_len': len(seq_cols),

    }


    scaler = None

    if num_cols:

        scaler = StandardScaler()

        train_num = scaler.fit_transform(train_df[num_cols].astype('float32'))

        train_num = np.asarray(train_num, dtype=np.float32)

    else:

        train_num = np.zeros((len(train_df), 0), dtype=np.float32)


    prep['scaler'] = scaler


    cat_cardinalities = []

    categories_by_col = {}

    for c in cat_cols:

        parts = [train_df[c].astype('string').fillna('__NA__')]

        if eval_df is not None:

            parts.append(eval_df[c].astype('string').fillna('__NA__'))

        if test_df is not None:

            parts.append(test_df[c].astype('string').fillna('__NA__'))

        categories = pd.Index(pd.concat(parts, ignore_index=True).unique())

        categories_by_col[c] = categories

        cat_cardinalities.append(int(len(categories)))

    prep['cat_cardinalities'] = cat_cardinalities


    def _build_split(df, num_array=None):

        if df is None:

            return None

        seq = np.asarray(df[seq_cols].astype('float32').to_numpy(copy=False), dtype=np.float32)[..., None]

        if num_cols:

            if num_array is None:

                num_array = scaler.transform(df[num_cols].astype('float32'))

            num = np.asarray(num_array, dtype=np.float32)

        else:

            num = np.zeros((len(df), 0), dtype=np.float32)

        if cat_cols:

            cat_list = []

            for c in cat_cols:

                codes = pd.Categorical(

                    df[c].astype('string').fillna('__NA__'),

                    categories=categories_by_col[c]

                ).codes.astype(np.int64)

                cat_list.append(codes)

            cat_arr = np.column_stack(cat_list).astype(np.int64, copy=False)

        else:

            cat_arr = np.zeros((len(df), 0), dtype=np.int64)

        out = {

            'seq': seq,

            'num': num,

            'cat': cat_arr,

            'wl9': np.asarray(df['water_level_9'].astype('float32').to_numpy(copy=False), dtype=np.float32),

        }

        if target_col in df.columns:

            out['y'] = np.asarray(df[target_col].astype('float32').to_numpy(copy=False), dtype=np.float32)

        return out


    prep['train'] = _build_split(train_df, train_num)

    prep['eval'] = _build_split(eval_df)

    prep['test'] = _build_split(test_df)

    return prep



# -----------------------------------------------------------------

# SCSE Block (Squeeze-and-Channel-Squeeze-and-Excitation)

# -----------------------------------------------------------------

class SCSEBlock(nn.Module):

    """Channel + Spatial Squeeze-and-Excitation for 1D feature vectors."""

    def __init__(self, channels, reduction=4):

        super().__init__()

        self.channel_se = nn.Sequential(

            nn.Linear(channels, channels // reduction),

            nn.ReLU(inplace=True),

            nn.Linear(channels // reduction, channels),

            nn.Sigmoid(),

        )

        self.spatial_se = nn.Sequential(

            nn.Linear(channels, 1),

            nn.Sigmoid(),

        )


    def forward(self, x):

        # x: (batch, channels)

        cse = self.channel_se(x) * x

        sse = self.spatial_se(x) * x

        return cse + sse



# -----------------------------------------------------------------

# GCN Preprocessor (no PyG required, pure PyTorch + scipy sparse)

# -----------------------------------------------------------------

from scipy.sparse import csr_matrix


def _build_adjacency_normalized(model_id, node_type):

    """Build normalized adjacency matrix D^{-1/2} A D^{-1/2} for GCN.

    Returns: (adj_norm as dense tensor, node_id_to_idx mapping, idx_to_node_id)

    """

    k = (model_id, node_type)

    if k not in EDGE_INDEX_CACHE:

        print(f"  WARNING: No edge index for ({model_id}, {node_type})")

        return None, None, None


    fc, tc, ei = EDGE_INDEX_CACHE[k]

    static = load_static(model_id, node_type)

    all_nids = sorted(static['node_id'].unique().astype(int))

    nid_map = {nid: i for i, nid in enumerate(all_nids)}

    n = len(all_nids)


    rows, cols = [], []

    for _, r in ei.iterrows():

        u, v = int(r[fc]), int(r[tc])

        if u in nid_map and v in nid_map:

            ui, vi = nid_map[u], nid_map[v]

            rows.extend([ui, vi])

            cols.extend([vi, ui])


    # Add self-loops

    rows.extend(range(n))

    cols.extend(range(n))

    vals = np.ones(len(rows), dtype='float32')

    A = csr_matrix((vals, (rows, cols)), shape=(n, n))


    # Normalize: D^{-1/2} A D^{-1/2}

    degrees = np.array(A.sum(axis=1)).flatten()

    D_inv_sqrt = np.where(degrees > 0, 1.0 / np.sqrt(degrees), 0)

    D_inv_sqrt_mat = csr_matrix(np.diag(D_inv_sqrt))

    A_norm = D_inv_sqrt_mat @ A @ D_inv_sqrt_mat


    return (torch.tensor(A_norm.toarray(), dtype=torch.float32),

            nid_map,

            {i: nid for nid, i in nid_map.items()})



class GCNLayer(nn.Module):

    """Simple GCN layer: X' = ReLU(A_norm @ X @ W)"""

    def __init__(self, in_dim, out_dim):

        super().__init__()

        self.linear = nn.Linear(in_dim, out_dim)

        self.bn = nn.BatchNorm1d(out_dim)


    def forward(self, x, adj):

        # x: (n_nodes, in_dim), adj: (n_nodes, n_nodes)

        h = adj @ x  # message passing

        h = self.linear(h)

        h = self.bn(h)

        return torch.relu(h)



class GCNPreprocessor(nn.Module):

    """2-layer GCN that produces node embeddings from static features."""

    def __init__(self, in_dim, hidden_dim=64, out_dim=32, n_layers=2):

        super().__init__()

        layers = []

        dims = [in_dim] + [hidden_dim] * (n_layers - 1) + [out_dim]

        for i in range(n_layers):

            layers.append(GCNLayer(dims[i], dims[i+1]))

        self.layers = nn.ModuleList(layers)

        self.out_dim = out_dim


    def forward(self, x, adj):

        for layer in self.layers:

            x = layer(x, adj)

        return x



def _compute_gcn_embeddings(train_df, eval_df, test_df, model_id, node_type,

                             gcn_hidden=64, gcn_out=32, gcn_layers=2,

                             n_epochs=100, lr=0.01):

    """Pre-compute GCN node embeddings using self-supervised learning.

    Train GCN to reconstruct node features from graph-aggregated features.

    Returns dict: node_id -> embedding vector.

    """

    adj_norm, nid_map, idx_to_nid = _build_adjacency_normalized(model_id, node_type)

    if adj_norm is None:

        return {}


    n_nodes = len(nid_map)


    # Collect per-node features: use warmup WLs + static features averaged across events

    warmup_cols = sorted(

        [c for c in train_df.columns if re.fullmatch(r'water_level_\d+', c)],

        key=lambda x: int(x.split('_')[-1])

    )

    # Static features per node (take first occurrence)

    static_cols = ['node_scale', 'node_warmup_range'] if 'node_scale' in train_df.columns else []

    # Add some graph-related static features

    for c in train_df.columns:

        if c in ('upstream_total', 'degree', 'betweenness', 'pagerank',

                 'pipe_cap_total_in', 'pipe_cap_total_out', 'position_x', 'position_y'):

            if c in train_df.columns:

                static_cols.append(c)


    feat_cols = warmup_cols + static_cols

    if not feat_cols:

        print("  WARNING: No features for GCN preprocessing")

        return {}


    # Average features per node across all events

    node_feats = train_df.groupby('node_id')[feat_cols].mean().reset_index()

    node_feat_matrix = np.zeros((n_nodes, len(feat_cols)), dtype='float32')

    for _, row in node_feats.iterrows():

        nid = int(row['node_id'])

        if nid in nid_map:

            node_feat_matrix[nid_map[nid]] = row[feat_cols].values.astype('float32')


    # Normalize

    feat_mean = node_feat_matrix.mean(axis=0, keepdims=True)

    feat_std = node_feat_matrix.std(axis=0, keepdims=True) + 1e-8

    node_feat_matrix = (node_feat_matrix - feat_mean) / feat_std


    X = torch.tensor(node_feat_matrix, dtype=torch.float32)


    # Train GCN with self-supervised objective: reconstruct features from neighbors

    gcn = GCNPreprocessor(len(feat_cols), gcn_hidden, gcn_out, gcn_layers)

    decoder = nn.Linear(gcn_out, len(feat_cols))

    optimizer = torch.optim.Adam(list(gcn.parameters()) + list(decoder.parameters()), lr=lr)

    criterion = nn.MSELoss()


    gcn.train()

    for epoch in range(n_epochs):

        optimizer.zero_grad()

        embeddings = gcn(X, adj_norm)

        reconstructed = decoder(embeddings)

        loss = criterion(reconstructed, X)

        loss.backward()

        optimizer.step()


    gcn.eval()

    with torch.no_grad():

        final_embeddings = gcn(X, adj_norm).numpy()


    # Build node_id -> embedding mapping

    result = {}

    for nid, idx in nid_map.items():

        result[nid] = final_embeddings[idx]


    print(f"  GCN embeddings: {n_nodes} nodes, {gcn_out}d, "

          f"final loss={loss.item():.6f}")


    return result, gcn_out



# -----------------------------------------------------------------

# Numeric Embeddings via rtdl_num_embeddings library

# pip install rtdl_num_embeddings

# -----------------------------------------------------------------

try:

    from rtdl_num_embeddings import PiecewiseLinearEncoding as _RtdlPLE

    from rtdl_num_embeddings import compute_bins as _rtdl_compute_bins

    RTDL_AVAILABLE = True

    print("  rtdl_num_embeddings available")

except ImportError:

    RTDL_AVAILABLE = False

    print("  WARNING: rtdl_num_embeddings not installed, NumEmbedder will use fallback")



class NumEmbedder(nn.Module):

    """Numeric feature embedding using rtdl PiecewiseLinearEncoding.

    Falls back to BatchNorm+Linear if rtdl not available.

    """

    def __init__(self, n_features, n_bins=48, d_out=256):

        super().__init__()

        self.n_features = n_features

        self.n_bins = n_bins

        self.d_out = d_out

        self.use_rtdl = RTDL_AVAILABLE


        if self.use_rtdl:

            # Will be initialized from data via initialize_from_data()

            self.ple = None  # created after seeing data

            self.proj = None  # created after knowing ple output dim

        else:

            self.fallback = nn.Sequential(

                nn.BatchNorm1d(n_features),

                nn.Linear(n_features, d_out),

                nn.ReLU(inplace=True),

                nn.Dropout(0.1),

            )


    def initialize_from_data(self, X):

        """Initialize PLE bins from training data."""

        if not self.use_rtdl:

            return

        X_tensor = torch.as_tensor(X, dtype=torch.float32)

        # Compute bin edges from data quantiles

        bins = _rtdl_compute_bins(X_tensor, n_bins=self.n_bins)

        self.ple = _RtdlPLE(bins)

        # PLE output: (batch, n_features, n_bins) -> flatten -> project

        ple_out_dim = self.n_features * self.n_bins

        self.proj = nn.Sequential(

            nn.Linear(ple_out_dim, self.d_out),

            nn.ReLU(inplace=True),

            nn.Dropout(0.1),

        )

        # Move to same device as other params

        device = next(self.parameters(), torch.tensor(0)).device

        self.ple = self.ple.to(device)

        self.proj = self.proj.to(device)


    def forward(self, x):

        if self.use_rtdl and self.ple is not None:

            # PLE returns (batch, n_features, n_bins)

            encoded = self.ple(x)

            # Flatten: (batch, n_features * n_bins)

            encoded = encoded.reshape(x.shape[0], -1)

            return self.proj(encoded)

        elif hasattr(self, 'fallback'):

            return self.fallback(x)

        else:

            # Safety fallback

            return x



# -----------------------------------------------------------------

# Sequence Encoders

# -----------------------------------------------------------------

class BiGRUEncoder(nn.Module):

    def __init__(self, input_dim, hidden, layers, dropout):

        super().__init__()

        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden,

                          num_layers=layers, batch_first=True,

                          bidirectional=True,

                          dropout=dropout if layers > 1 else 0.0)

        self.out_dim = hidden * 2


    def forward(self, x):

        _, h_n = self.gru(x)

        return torch.cat([h_n[-2], h_n[-1]], dim=1)



class LSTMEncoder(nn.Module):

    def __init__(self, input_dim, hidden, layers, dropout):

        super().__init__()

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden,

                            num_layers=layers, batch_first=True,

                            bidirectional=True,

                            dropout=dropout if layers > 1 else 0.0)

        self.out_dim = hidden * 2


    def forward(self, x):

        _, (h_n, _) = self.lstm(x)

        return torch.cat([h_n[-2], h_n[-1]], dim=1)



class MultiScaleCNNEncoder(nn.Module):

    def __init__(self, input_dim, hidden=64):

        super().__init__()

        self.conv_short = nn.Conv1d(input_dim, hidden, kernel_size=3, padding=1)

        self.conv_mid = nn.Conv1d(input_dim, hidden, kernel_size=7, padding=3)

        self.conv_long = nn.Conv1d(input_dim, hidden, kernel_size=15, padding=7)

        self.bn = nn.BatchNorm1d(hidden * 3)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.out_dim = hidden * 3


    def forward(self, x):

        # x: (batch, seq_len, channels) -> (batch, channels, seq_len)

        x = x.transpose(1, 2)

        c1 = torch.relu(self.conv_short(x))

        c2 = torch.relu(self.conv_mid(x))

        c3 = torch.relu(self.conv_long(x))

        cat = torch.cat([c1, c2, c3], dim=1)  # (batch, hidden*3, seq_len)

        cat = self.bn(cat)

        pooled = self.pool(cat).squeeze(-1)  # (batch, hidden*3)

        return pooled



class TransformerEncoder_(nn.Module):

    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, max_seq_len=256):

        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        self.pos_enc = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=d_model, nhead=nhead,

            dim_feedforward=d_model * 2,

            dropout=0.1, batch_first=True

        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.out_dim = d_model


    def forward(self, x):

        # x: (batch, seq_len, channels)

        batch_size, seq_len, _ = x.shape

        x = self.input_proj(x)

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)

        x = x + self.pos_enc(positions)

        encoded = self.encoder(x)

        # Mean pooling over sequence

        return encoded.mean(dim=1)



class BiGRUAttentionEncoder(nn.Module):

    """BiGRU with temporal attention over hidden states.

    Instead of using only the final hidden state, attention learns

    which timesteps are most important (e.g., rain peaks).

    """

    def __init__(self, input_dim, hidden, layers, dropout):

        super().__init__()

        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden,

                          num_layers=layers, batch_first=True,

                          bidirectional=True,

                          dropout=dropout if layers > 1 else 0.0)

        gru_out_dim = hidden * 2  # bidirectional

        # Attention: score each timestep

        self.attn_query = nn.Linear(gru_out_dim, gru_out_dim)

        self.attn_key = nn.Linear(gru_out_dim, gru_out_dim)

        self.attn_v = nn.Linear(gru_out_dim, 1)

        self.attn_dropout = nn.Dropout(dropout)

        self.out_dim = gru_out_dim


    def forward(self, x):

        # x: (batch, seq_len, input_dim)

        output, h_n = self.gru(x)

        # output: (batch, seq_len, hidden*2)

        # Compute attention scores

        q = torch.tanh(self.attn_query(output))  # (batch, seq_len, hidden*2)

        k = torch.tanh(self.attn_key(output))    # (batch, seq_len, hidden*2)

        scores = self.attn_v(q * k).squeeze(-1)  # (batch, seq_len)

        attn_weights = torch.softmax(scores, dim=1)  # (batch, seq_len)

        attn_weights = self.attn_dropout(attn_weights)

        # Weighted sum of GRU outputs

        context = (output * attn_weights.unsqueeze(-1)).sum(dim=1)  # (batch, hidden*2)

        return context



# -----------------------------------------------------------------

# Pre-Norm Transformer Encoder (bigger capacity, pre-LN, RoPE)

# -----------------------------------------------------------------

class RotaryPositionalEmbedding(nn.Module):

    """Rotary Positional Embedding (RoPE) for attention."""

    def __init__(self, d_model, max_seq_len=512):

        super().__init__()

        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))

        self.register_buffer('inv_freq', inv_freq)

        self.max_seq_len = max_seq_len


    def forward(self, seq_len, device):

        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)

        freqs = torch.einsum('i,j->ij', t, self.inv_freq)

        emb = torch.cat([freqs, freqs], dim=-1)

        return emb.cos(), emb.sin()



def _rotate_half(x):

    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]

    return torch.cat([-x2, x1], dim=-1)



def _apply_rope(x, cos, sin):

    # x: (batch, nhead, seq, d_head)

    return x * cos.unsqueeze(0).unsqueeze(0) + _rotate_half(x) * sin.unsqueeze(0).unsqueeze(0)



class PreNormTransformerBlock(nn.Module):

    """Pre-LayerNorm Transformer block with optional RoPE."""

    def __init__(self, d_model, nhead, ff_mult=4, dropout=0.1):

        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)

        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

        self.norm2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(

            nn.Linear(d_model, d_model * ff_mult),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(d_model * ff_mult, d_model),

            nn.Dropout(dropout),

        )


    def forward(self, x):

        # Pre-norm attention

        normed = self.norm1(x)

        x = x + self.attn(normed, normed, normed, need_weights=False)[0]

        # Pre-norm FFN

        x = x + self.ff(self.norm2(x))

        return x



class PreNormTransformerEncoder(nn.Module):

    """Larger Transformer with Pre-LayerNorm and learned positional embeddings."""

    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=4,

                 ff_mult=4, max_seq_len=512, dropout=0.1):

        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        self.pos_enc = nn.Embedding(max_seq_len, d_model)

        self.blocks = nn.ModuleList([

            PreNormTransformerBlock(d_model, nhead, ff_mult, dropout)

            for _ in range(num_layers)

        ])

        self.final_norm = nn.LayerNorm(d_model)

        self.out_dim = d_model


    def forward(self, x):

        batch_size, seq_len, _ = x.shape

        x = self.input_proj(x)

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)

        x = x + self.pos_enc(positions)

        for block in self.blocks:

            x = block(x)

        x = self.final_norm(x)

        return x.mean(dim=1)



# -----------------------------------------------------------------

# Unified FloodDLModel

# -----------------------------------------------------------------

class FloodDLModel(nn.Module):

    def __init__(self, config, num_dim, cat_cardinalities, seq_input_dim):

        super().__init__()

        arch = config['arch']

        dropout = config.get('dropout', 0.15)

        tab_hidden = config.get('tab_hidden', 256)

        head_hidden = config.get('head_hidden', 256)


        # 1. Sequence encoder

        if arch == 'bigru':

            self.seq_encoder = BiGRUEncoder(

                seq_input_dim, config.get('seq_hidden', 96),

                config.get('seq_layers', 2), dropout)

        elif arch == 'bigru_attn':

            self.seq_encoder = BiGRUAttentionEncoder(

                seq_input_dim, config.get('seq_hidden', 96),

                config.get('seq_layers', 2), dropout)

        elif arch == 'lstm':

            self.seq_encoder = LSTMEncoder(

                seq_input_dim, config.get('seq_hidden', 96),

                config.get('seq_layers', 2), dropout)

        elif arch == 'cnn':

            self.seq_encoder = MultiScaleCNNEncoder(

                seq_input_dim, config.get('cnn_hidden', 64))

        elif arch == 'transformer':

            self.seq_encoder = TransformerEncoder_(

                seq_input_dim, config.get('d_model', 64),

                config.get('nhead', 4), config.get('tf_layers', 2))

        elif arch == 'transformer_prenorm':

            self.seq_encoder = PreNormTransformerEncoder(

                seq_input_dim,

                d_model=config.get('d_model', 128),

                nhead=config.get('nhead', 8),

                num_layers=config.get('tf_layers', 4),

                ff_mult=config.get('tf_ff_mult', 4),

                dropout=dropout,

            )

        else:

            raise ValueError(f"Unknown arch: {arch}")


        seq_out_dim = self.seq_encoder.out_dim


        # 2. Numeric features

        self.num_dim = num_dim

        self.use_num_embed = config.get('use_num_embed', False)

        if num_dim > 0:

            if self.use_num_embed:

                self.num_block = NumEmbedder(

                    num_dim,

                    n_bins=config.get('num_embed_bins', 48),

                    d_out=tab_hidden

                )

                num_out_dim = tab_hidden

            else:

                self.num_block = nn.Sequential(

                    nn.BatchNorm1d(num_dim),

                    nn.Linear(num_dim, tab_hidden),

                    nn.ReLU(inplace=True),

                    nn.Dropout(dropout),

                )

                num_out_dim = tab_hidden

        else:

            self.num_block = None

            num_out_dim = 0


        # 3. Categorical embeddings

        self.cat_embeddings = nn.ModuleList([

            nn.Embedding(card + 1, _embedding_dim(card + 1))

            for card in cat_cardinalities

        ])

        cat_dim = sum(emb.embedding_dim for emb in self.cat_embeddings)


        # 4. Fusion

        fusion_dim = seq_out_dim + num_out_dim + cat_dim


        # 5. Optional SCSE

        self.use_scse = config.get('use_scse', False)

        if self.use_scse:

            self.scse = SCSEBlock(fusion_dim, config.get('scse_reduction', 4))


        # 6. Head

        self.head = nn.Sequential(

            nn.Linear(fusion_dim, head_hidden),

            nn.ReLU(inplace=True),

            nn.Dropout(dropout),

            nn.Linear(head_hidden, head_hidden // 2),

            nn.ReLU(inplace=True),

            nn.Dropout(dropout),

            nn.Linear(head_hidden // 2, 1),

        )


    def forward(self, seq_x, num_x, cat_x):

        seq_repr = self.seq_encoder(seq_x)

        parts = [seq_repr]


        if self.num_block is not None:

            parts.append(self.num_block(num_x))


        if len(self.cat_embeddings) > 0:

            cat_parts = []

            for i, emb in enumerate(self.cat_embeddings):

                cat_parts.append(emb(cat_x[:, i].clamp(min=0)))

            parts.append(torch.cat(cat_parts, dim=1))


        fused = torch.cat(parts, dim=1)


        if self.use_scse:

            fused = self.scse(fused)


        return self.head(fused).squeeze(-1)



# -----------------------------------------------------------------

# DataLoader helpers

# -----------------------------------------------------------------

class FloodDataset(torch.utils.data.Dataset):

    def __init__(self, data_dict, has_target=True):

        self.seq = torch.from_numpy(data_dict['seq'])

        self.num = torch.from_numpy(data_dict['num'])

        self.cat = torch.from_numpy(data_dict['cat'])

        self.has_target = has_target

        if has_target and 'y' in data_dict:

            self.y = torch.from_numpy(data_dict['y'])

        else:

            self.has_target = False


    def __len__(self):

        return len(self.seq)


    def __getitem__(self, idx):

        if self.has_target:

            return self.seq[idx], self.num[idx], self.cat[idx], self.y[idx]

        return self.seq[idx], self.num[idx], self.cat[idx]



def _make_train_loader(data_dict, batch_size=1024, num_workers=0, shuffle=True):

    ds = FloodDataset(data_dict, has_target=True)

    return torch.utils.data.DataLoader(ds, batch_size=batch_size,

                                        shuffle=shuffle, num_workers=num_workers,

                                        pin_memory=True, drop_last=False)



def _make_pred_loader(data_dict, batch_size=16384, num_workers=0):

    ds = FloodDataset(data_dict, has_target=False)

    return torch.utils.data.DataLoader(ds, batch_size=batch_size,

                                        shuffle=False, num_workers=num_workers,

                                        pin_memory=True, drop_last=False)



@torch.no_grad()

def _predict_dl(model, loader, device):

    model.eval()

    preds = []

    for batch in loader:

        seq_x, num_x, cat_x = batch[0], batch[1], batch[2]

        seq_x = seq_x.to(device, non_blocking=True)

        num_x = num_x.to(device, non_blocking=True)

        cat_x = cat_x.to(device, non_blocking=True)

        pred = model(seq_x, num_x, cat_x)

        preds.append(pred.detach().float().cpu().numpy())

    return np.concatenate(preds, axis=0)



# -----------------------------------------------------------------

# Training functions

# -----------------------------------------------------------------

def _train_dl_once(prep, config, seed, use_delta=True, verbose_prefix=''):

    """Train a FloodDLModel with early stopping on validation."""

    if prep.get('eval') is None:

        raise ValueError("prep['eval'] is required for validation training")


    seed_everything(seed)

    device = torch.device(config['device'])


    model = FloodDLModel(

        config=config,

        num_dim=prep['train']['num'].shape[1],

        cat_cardinalities=prep['cat_cardinalities'],

        seq_input_dim=prep['train']['seq'].shape[-1],

    ).to(device)


    # Initialize NumEmbedder from data if needed

    if config.get('use_num_embed', False) and hasattr(model.num_block, 'initialize_from_data'):

        model.num_block.initialize_from_data(prep['train']['num'])


    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],

                                  weight_decay=config.get('weight_decay', 1e-5))


    # Cosine annealing scheduler

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

        optimizer, T_max=config['epochs'], eta_min=config['lr'] * 0.01)


    criterion = nn.MSELoss()


    train_loader = _make_train_loader(

        prep['train'], batch_size=config['batch_size'],

        num_workers=config.get('num_workers', 0), shuffle=True)

    valid_loader = _make_pred_loader(

        prep['eval'], batch_size=config.get('eval_batch_size', 16384),

        num_workers=config.get('num_workers', 0))


    best_rmse = np.inf

    best_epoch = -1

    best_state = None

    patience_left = config['patience']


    for epoch in range(1, config['epochs'] + 1):

        model.train()

        t0 = time.time()

        running_loss = 0.0

        n_obs = 0


        for batch in train_loader:

            seq_x, num_x, cat_x, y = batch

            seq_x = seq_x.to(device, non_blocking=True)

            num_x = num_x.to(device, non_blocking=True)

            cat_x = cat_x.to(device, non_blocking=True)

            y = y.to(device, non_blocking=True)


            optimizer.zero_grad(set_to_none=True)

            pred = model(seq_x, num_x, cat_x)

            loss = criterion(pred, y)

            loss.backward()


            gc_val = config.get('grad_clip', 0)

            if gc_val and gc_val > 0:

                nn.utils.clip_grad_norm_(model.parameters(), gc_val)


            optimizer.step()


            bs = y.shape[0]

            running_loss += float(loss.detach().item()) * bs

            n_obs += bs


        scheduler.step()


        train_rmse_target = float(np.sqrt(max(running_loss / max(n_obs, 1), 0.0)))


        valid_pred = _predict_dl(model, valid_loader, device)

        valid_rmse_val = rmse(prep['eval']['y'], valid_pred)

        elapsed = time.time() - t0

        prefix = f"{verbose_prefix} | " if verbose_prefix else ""

        print(f"{prefix}Epoch {epoch:03d} | train_rmse={train_rmse_target:.6f} | "

              f"valid_rmse={valid_rmse_val:.6f} | lr={optimizer.param_groups[0]['lr']:.2e} | {elapsed:.1f}s")


        if valid_rmse_val < best_rmse:

            best_rmse = valid_rmse_val

            best_epoch = epoch

            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            patience_left = config['patience']

        else:

            patience_left -= 1

            if patience_left <= 0:

                print(f"{prefix}Early stopping at epoch {epoch}")

                break


    model.load_state_dict(best_state)

    final_pred = _predict_dl(model, valid_loader, device)

    if use_delta:

        final_pred = final_pred + prep['eval']['wl9']


    return {

        'model': model,

        'best_epoch': int(best_epoch),

        'best_rmse': float(best_rmse),

        'valid_pred': final_pred.astype(np.float32),

        'state_dict_cpu': best_state,

    }



def _train_dl_fixed_epochs(prep, config, seed, n_epochs, log_prefix=''):

    """Train without validation (for refit on full data)."""

    seed_everything(seed)

    device = torch.device(config['device'])


    model = FloodDLModel(

        config=config,

        num_dim=prep['train']['num'].shape[1],

        cat_cardinalities=prep['cat_cardinalities'],

        seq_input_dim=prep['train']['seq'].shape[-1],

    ).to(device)


    if config.get('use_num_embed', False) and hasattr(model.num_block, 'initialize_from_data'):

        model.num_block.initialize_from_data(prep['train']['num'])


    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],

                                  weight_decay=config.get('weight_decay', 1e-5))

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

        optimizer, T_max=int(n_epochs), eta_min=config['lr'] * 0.01)

    criterion = nn.MSELoss()


    train_loader = _make_train_loader(

        prep['train'], batch_size=config['batch_size'],

        num_workers=config.get('num_workers', 0), shuffle=True)


    for epoch in range(1, int(n_epochs) + 1):

        model.train()

        running_loss = 0.0

        n_obs = 0

        t0 = time.time()


        for batch in train_loader:

            seq_x, num_x, cat_x, y = batch

            seq_x = seq_x.to(device, non_blocking=True)

            num_x = num_x.to(device, non_blocking=True)

            cat_x = cat_x.to(device, non_blocking=True)

            y = y.to(device, non_blocking=True)


            optimizer.zero_grad(set_to_none=True)

            pred = model(seq_x, num_x, cat_x)

            loss = criterion(pred, y)

            loss.backward()


            gc_val = config.get('grad_clip', 0)

            if gc_val and gc_val > 0:

                nn.utils.clip_grad_norm_(model.parameters(), gc_val)


            optimizer.step()


            bs = y.shape[0]

            running_loss += float(loss.detach().item()) * bs

            n_obs += bs


        scheduler.step()

        train_rmse = float(np.sqrt(max(running_loss / max(n_obs, 1), 0.0)))

        elapsed = time.time() - t0

        prefix = f"{log_prefix} | " if log_prefix else ""

        print(f"{prefix}Epoch {epoch:03d}/{int(n_epochs):03d} | train_rmse={train_rmse:.6f} | {elapsed:.1f}s")


    return model



def _run_dl_groupkfold(refit_df, test_df, feature_cols, cat_cols, target_col,

                        config, seed, use_delta=True, n_splits=5,

                        past_w=120, future_w=10):

    """GroupKFold training for test-time DL ensemble."""

    n_unique_events = refit_df['event_id'].nunique()

    n_splits = int(min(max(2, n_splits), n_unique_events))


    gkf = GroupKFold(n_splits=n_splits)

    fold_rows = []

    test_pred_list = []


    for fold, (tr_idx, va_idx) in enumerate(gkf.split(refit_df, groups=refit_df['event_id']), start=1):

        fold_train = refit_df.iloc[tr_idx].reset_index(drop=True)

        fold_valid = refit_df.iloc[va_idx].reset_index(drop=True)


        if config.get('use_rain_window', False):

            fold_prep = _prepare_rain_window_inputs(

                train_df=fold_train, eval_df=fold_valid, test_df=test_df,

                feature_cols=feature_cols, cat_cols=cat_cols,

                target_col=target_col, past_w=past_w, future_w=future_w)

        else:

            fold_prep = _prepare_bigru_inputs(

                train_df=fold_train, eval_df=fold_valid, test_df=test_df,

                feature_cols=feature_cols, cat_cols=cat_cols, target_col=target_col)


        result = _train_dl_once(

            prep=fold_prep, config=config, seed=seed,

            use_delta=use_delta,

            verbose_prefix=f"GKF seed={seed} fold={fold}")


        test_loader = _make_pred_loader(

            fold_prep['test'],

            batch_size=config.get('eval_batch_size', 16384),

            num_workers=config.get('num_workers', 0))

        device = torch.device(config['device'])

        test_pred = _predict_dl(result['model'], test_loader, device)

        if use_delta:

            test_pred = test_pred + fold_prep['test']['wl9']

        test_pred_list.append(test_pred.astype('float64'))


        fold_rows.append({

            'seed': seed, 'fold': fold,

            'best_epoch': result['best_epoch'],

            'rmse': result['best_rmse'],

        })


        del result, fold_prep

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    fold_df = pd.DataFrame(fold_rows)


    if not test_pred_list:

        raise RuntimeError("No GroupKFold models were trained")


    test_pred_ensemble = np.mean(np.column_stack(test_pred_list), axis=1).astype(np.float32)


    summary = {

        'seed': seed,

        'n_folds': int(len(fold_df)),

        'epoch_mean': float(fold_df['best_epoch'].mean()),

        'epoch_median': float(fold_df['best_epoch'].median()),

        'rmse_mean': float(fold_df['rmse'].mean()),

        'rmse_std': float(fold_df['rmse'].std(ddof=0)) if len(fold_df) > 1 else 0.0,

    }


    return summary, fold_df, test_pred_ensemble



# Reload RAIN_CACHE (cleared by cell 12 after building frames)

print("\nReloading rain cache for rain-window construction...")

all_event_dirs = (list(train_pool.get(m, [])) +

                  list(valid_pool.get(m, [])) +

                  list(test_events.get(m, [])))

for event_dir in all_event_dirs:

    load_rain(m, event_dir)

print(f"  RAIN_CACHE populated: {len(RAIN_CACHE)} events")


# Prepare rain-window inputs for validation

print("Preparing rain-window inputs for validation...")

rain_window_prep = _prepare_rain_window_inputs(

    train_df=train_df.fillna(0),

    eval_df=eval_df.fillna(0),

    test_df=test_df.fillna(0),

    feature_cols=feat,

    cat_cols=cat,

    target_col=target,

    past_w=120,

    future_w=10,

)


# Also prepare legacy inputs for baseline comparison

bigru_valid_artifacts = _prepare_bigru_inputs(

    train_df=train_df.fillna(0),

    eval_df=eval_df.fillna(0),

    test_df=test_df.fillna(0),

    feature_cols=feat,

    cat_cols=cat,

    target_col=target,

)


print("BiGRU sequence cols:", bigru_valid_artifacts.get('seq_cols', bigru_valid_artifacts.get('warmup_cols', []))[:3], "...")

print("Rain-window num cols:", len(rain_window_prep['num_cols']))

print("Rain-window cat cols:", rain_window_prep['cat_cols'])

print("Rain-window seq shape:", rain_window_prep['train']['seq'].shape)


# ---------------------------------------------------------------

# GCN pre-computation: add graph-aware embeddings as extra features

# ---------------------------------------------------------------

gcn_prep_available = False

rain_window_prep_gcn = None

feat_gcn = list(feat)  # default: same as feat, extended if GCN works


# Check if any experiment uses GCN

any_gcn = any(cfg.get('use_gcn', False) for cfg in DL_EXPERIMENTS.values())


if any_gcn:

    print("\nComputing GCN node embeddings...")

    try:

        gcn_cfg = next(cfg for cfg in DL_EXPERIMENTS.values() if cfg.get('use_gcn'))

        gcn_result = _compute_gcn_embeddings(

            train_df, eval_df, test_df,

            model_id=m, node_type=nt,

            gcn_hidden=gcn_cfg.get('gcn_hidden', 64),

            gcn_out=gcn_cfg.get('gcn_out', 32),

            gcn_layers=gcn_cfg.get('gcn_layers', 2),

            n_epochs=gcn_cfg.get('gcn_epochs', 100),

        )

        if gcn_result and isinstance(gcn_result, tuple) and len(gcn_result[0]) > 0:

            gcn_emb_dict, gcn_dim = gcn_result

            gcn_col_names = [f'gcn_emb_{i}' for i in range(gcn_dim)]


            # Merge GCN embeddings into dataframes

            for df_part in [train_df, eval_df, test_df]:

                gcn_arr = np.zeros((len(df_part), gcn_dim), dtype='float32')

                nids = df_part['node_id'].values

                for i, nid in enumerate(nids):

                    nid_int = int(nid)

                    if nid_int in gcn_emb_dict:

                        gcn_arr[i] = gcn_emb_dict[nid_int]

                for j, col_name in enumerate(gcn_col_names):

                    df_part[col_name] = gcn_arr[:, j]


            # Build extended feature list with GCN columns

            feat_gcn = list(feat) + gcn_col_names


            # Build rain_window_prep_gcn with GCN features included

            print("Preparing rain-window inputs WITH GCN embeddings...")

            rain_window_prep_gcn = _prepare_rain_window_inputs(

                train_df=train_df.fillna(0),

                eval_df=eval_df.fillna(0),

                test_df=test_df.fillna(0),

                feature_cols=feat_gcn,

                cat_cols=cat,

                target_col=target,

                past_w=120,

                future_w=10,

            )

            gcn_prep_available = True

            print(f"  GCN prep: num_cols={len(rain_window_prep_gcn['num_cols'])} "

                  f"(+{gcn_dim} GCN features)")

        else:

            print("  WARNING: GCN embeddings empty, skipping GCN prep")

    except Exception as e:

        print(f"  WARNING: GCN pre-computation failed: {e}")

        import traceback; traceback.print_exc()


print("Ready for experiments.")


=== Model 2, NodeType 1 — CatBoost + Multi-DL ensemble ===
  Features: 264, Rows: train=2,545,884 valid=852,390 test=1,422,036
  Target: wl_delta_target
  DL experiments: ['N5_big_tf_w200', 'N2_transformer_scse', 'N3_transformer_gcn']
  rtdl_num_embeddings available

Reloading rain cache for rain-window construction...
  RAIN_CACHE populated: 99 events
Preparing rain-window inputs for validation...
  Building rain-window sequences (past=120, future=10, seq_len=141, channels=3)...
  Rain-window sequences built in 26.7s
  Train seq shape: (2545884, 141, 3)
  Num cols: 242, Cat cols: 1
  Verification: seq diff between timesteps of node 0: 2.9450 (OK - different)
BiGRU sequence cols: ['water_level_0', 'water_level_1', 'water_level_2'] ...
Rain-window num cols: 242
Rain-window cat cols: ['node_id_cat']
Rain-window seq shape: (2545884, 141, 3)

Computing GCN node embeddings...
  GCN embeddings: 198 nodes, 32d, final loss=0.026077
Preparing rain-window inputs WITH GCN embeddings...
  Buildin

In [16]:
# =========================================================

# REFIT TOP-3 DL MODELS + INFERENCE

# =========================================================

import gc


refit_df = pd.concat([train_df, eval_df], ignore_index=True)

print(f"refit_df: {refit_df.shape}")


# Ensure feat_gcn is available

if 'feat_gcn' not in dir():

    feat_gcn = feat


# Ensure RAIN_CACHE is populated for GroupKFold

if len(RAIN_CACHE) == 0:

    print("Reloading rain cache for GroupKFold...")

    for event_dir in (list(train_pool.get(m, [])) +

                      list(valid_pool.get(m, [])) +

                      list(test_events.get(m, []))):

        load_rain(m, event_dir)

    print(f"  RAIN_CACHE: {len(RAIN_CACHE)} events")


# --- Top-3 experiments to refit ---

REFIT_MODELS = ['N5_big_tf_w200', 'N3_transformer_gcn', 'N2_transformer_scse']

REFIT_SEEDS = [56]  # 3 seeds for seed blending

GROUPKFOLD_SPLITS = 5


all_model_preds = {}  # model_name -> averaged test predictions


for model_name in REFIT_MODELS:

    print(f"\n{'='*70}")

    print(f"  REFIT: {model_name}")

    print(f"{'='*70}")


    exp_config = DL_EXPERIMENTS[model_name]

    past_w = exp_config.get('past_window', 120)

    future_w = exp_config.get('future_window', 10)


    # Use extended feature list for GCN experiments

    exp_feat = feat_gcn if (exp_config.get('use_gcn', False) and gcn_prep_available) else feat


    seed_preds = []


    for seed in REFIT_SEEDS:

        print(f"\n--- {model_name} GroupKFold seed={seed} ---")

        t0 = time.time()


        gkf_summary, gkf_fold_df, gkf_test_pred = _run_dl_groupkfold(

            refit_df=refit_df.fillna(0),

            test_df=test_df.fillna(0),

            feature_cols=exp_feat,

            cat_cols=cat,

            target_col=target,

            config=exp_config,

            seed=seed,

            use_delta=use_delta,

            n_splits=GROUPKFOLD_SPLITS,

            past_w=past_w,

            future_w=future_w,

        )


        seed_preds.append(gkf_test_pred.astype('float64'))

        elapsed = time.time() - t0

        print(f"  {model_name} seed={seed}: fold_rmse_mean={gkf_summary['rmse_mean']:.6f} "

              f"+/- {gkf_summary['rmse_std']:.6f}, time={elapsed:.0f}s")

        display(gkf_fold_df)


        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    # Average across seeds

    avg_pred = np.mean(np.column_stack(seed_preds), axis=1)

    all_model_preds[model_name] = avg_pred

    print(f"\n  {model_name}: seed-averaged pred range = [{avg_pred.min():.4f}, {avg_pred.max():.4f}]")


print(f"\nAll {len(REFIT_MODELS)} models refit complete.")



# --- Create Submission Files ---

print("\nCreating final submission files...")



for model_name in REFIT_MODELS:

    pred_vals = all_model_preds[model_name]

    out_df = test_df[KEY_COLS].copy()

    out_df['water_level'] = pred_vals.astype('float64')

    fname = f"submission_m2_nt1_{model_name}.parquet"

    out_df.to_parquet(fname, index=False)

    print(f"  Saved {fname}: mean={out_df['water_level'].mean():.4f}")



# --- Also export a simple average ensemble of all 3 ---

ens_pred = np.mean(

    np.column_stack([all_model_preds[m] for m in REFIT_MODELS]),

    axis=1

)

out_ens = test_df[KEY_COLS].copy()

out_ens['water_level'] = ens_pred.astype('float64')

out_ens.to_parquet("submission_m2_nt1_v2_dl_ensemble_top3.parquet", index=False)

print(f"\n  Saved submission_m2_nt1_v2_dl_ensemble_top3.parquet: {out_ens.shape}")

print(f"    water_level: mean={out_ens['water_level'].mean():.4f}, "

      f"std={out_ens['water_level'].std():.4f}")


print("\n=== DONE ===")

print(f"Files created:")

for model_name in REFIT_MODELS:

    print(f"  - submission_m2_nt1_{model_name}.parquet")

print(f"  - submission_m2_nt1_v2_dl_ensemble_top3.parquet")

refit_df: (3398274, 302)

  REFIT: N5_big_tf_w200

--- N5_big_tf_w200 GroupKFold seed=56 ---
  Building rain-window sequences (past=200, future=10, seq_len=221, channels=3)...
  Rain-window sequences built in 38.3s
  Train seq shape: (2722896, 221, 3)
  Num cols: 242, Cat cols: 1
  Verification: seq diff between timesteps of node 0: 2.9450 (OK - different)
GKF seed=56 fold=1 | Epoch 001 | train_rmse=0.932519 | valid_rmse=0.466731 | lr=1.50e-04 | 114.7s
GKF seed=56 fold=1 | Epoch 002 | train_rmse=0.529435 | valid_rmse=0.403850 | lr=1.49e-04 | 121.4s
GKF seed=56 fold=1 | Epoch 003 | train_rmse=0.476898 | valid_rmse=0.371909 | lr=1.48e-04 | 113.1s
GKF seed=56 fold=1 | Epoch 004 | train_rmse=0.449047 | valid_rmse=0.364846 | lr=1.46e-04 | 116.5s
GKF seed=56 fold=1 | Epoch 005 | train_rmse=0.426990 | valid_rmse=0.344443 | lr=1.44e-04 | 114.8s
GKF seed=56 fold=1 | Epoch 006 | train_rmse=0.412330 | valid_rmse=0.351465 | lr=1.42e-04 | 121.3s
GKF seed=56 fold=1 | Epoch 007 | train_rmse=0.399820 

,seed,fold,best_epoch,rmse
0,56,1,37,0.313846
1,56,2,20,0.340535
2,56,3,5,0.363754
3,56,4,18,0.310926
4,56,5,3,0.499245



  N5_big_tf_w200: seed-averaged pred range = [22.8738, 47.9106]

  REFIT: N3_transformer_gcn

--- N3_transformer_gcn GroupKFold seed=56 ---
  Building rain-window sequences (past=120, future=10, seq_len=141, channels=3)...
  Rain-window sequences built in 27.4s
  Train seq shape: (2722896, 141, 3)
  Num cols: 274, Cat cols: 1
  Verification: seq diff between timesteps of node 0: 2.9450 (OK - different)
GKF seed=56 fold=1 | Epoch 001 | train_rmse=0.986172 | valid_rmse=0.477749 | lr=1.50e-04 | 89.6s
GKF seed=56 fold=1 | Epoch 002 | train_rmse=0.539724 | valid_rmse=0.406973 | lr=1.49e-04 | 89.7s
GKF seed=56 fold=1 | Epoch 003 | train_rmse=0.486744 | valid_rmse=0.384694 | lr=1.48e-04 | 89.5s
GKF seed=56 fold=1 | Epoch 004 | train_rmse=0.458513 | valid_rmse=0.381587 | lr=1.46e-04 | 89.1s
GKF seed=56 fold=1 | Epoch 005 | train_rmse=0.437017 | valid_rmse=0.388392 | lr=1.44e-04 | 89.2s
GKF seed=56 fold=1 | Epoch 006 | train_rmse=0.423183 | valid_rmse=0.370367 | lr=1.42e-04 | 89.4s
GKF seed=56

,seed,fold,best_epoch,rmse
0,56,1,22,0.330343
1,56,2,11,0.400524
2,56,3,26,0.343839
3,56,4,9,0.389112
4,56,5,30,0.385516



  N3_transformer_gcn: seed-averaged pred range = [22.8989, 48.0087]

  REFIT: N2_transformer_scse

--- N2_transformer_scse GroupKFold seed=56 ---
  Building rain-window sequences (past=120, future=10, seq_len=141, channels=3)...
  Rain-window sequences built in 26.6s
  Train seq shape: (2722896, 141, 3)
  Num cols: 242, Cat cols: 1
  Verification: seq diff between timesteps of node 0: 2.9450 (OK - different)
GKF seed=56 fold=1 | Epoch 001 | train_rmse=0.982707 | valid_rmse=0.463782 | lr=1.50e-04 | 84.5s
GKF seed=56 fold=1 | Epoch 002 | train_rmse=0.545390 | valid_rmse=0.447516 | lr=1.49e-04 | 84.6s
GKF seed=56 fold=1 | Epoch 003 | train_rmse=0.481968 | valid_rmse=0.383607 | lr=1.48e-04 | 84.6s
GKF seed=56 fold=1 | Epoch 004 | train_rmse=0.448994 | valid_rmse=0.381652 | lr=1.46e-04 | 84.7s
GKF seed=56 fold=1 | Epoch 005 | train_rmse=0.425284 | valid_rmse=0.360690 | lr=1.44e-04 | 84.5s
GKF seed=56 fold=1 | Epoch 006 | train_rmse=0.407819 | valid_rmse=0.375486 | lr=1.42e-04 | 84.5s
GKF s

,seed,fold,best_epoch,rmse
0,56,1,15,0.342250
1,56,2,13,0.371828
2,56,3,20,0.320709
3,56,4,8,0.391639
4,56,5,36,0.377227



  N2_transformer_scse: seed-averaged pred range = [22.9367, 47.9736]

All 3 models refit complete.

Creating final submission files...
  Saved submission_m2_nt1_N5_big_tf_w200.parquet: mean=39.9006
  Saved submission_m2_nt1_N3_transformer_gcn.parquet: mean=39.9224
  Saved submission_m2_nt1_N2_transformer_scse.parquet: mean=39.9298

  Saved submission_m2_nt1_v2_dl_ensemble_top3.parquet: (1422036, 5)
    water_level: mean=39.9176, std=3.2225

=== DONE ===
Files created:
  - submission_m2_nt1_N5_big_tf_w200.parquet
  - submission_m2_nt1_N3_transformer_gcn.parquet
  - submission_m2_nt1_N2_transformer_scse.parquet
  - submission_m2_nt1_v2_dl_ensemble_top3.parquet


In [17]:
# =========================================================

# EXPORT PARQUET FILES

# =========================================================

KEY_COLS = ["model_id", "event_id", "node_type", "node_id"]


for model_name, preds in all_model_preds.items():

    out_df = test_df[KEY_COLS].copy()

    out_df['water_level'] = preds.astype('float64')


    # Sanity checks

    assert len(out_df) == len(test_df), f"Row count mismatch: {len(out_df)} vs {len(test_df)}"

    assert out_df['water_level'].isna().sum() == 0, "NaN in predictions!"

    assert out_df['water_level'].std() > 0.01, "Predictions look constant!"


    fname = f"submission_m2_nt1_{model_name}.parquet"

    out_df.to_parquet(fname, index=False)

    print(f"  Saved {fname}: {out_df.shape}")

    print(f"    water_level: mean={out_df['water_level'].mean():.4f}, "

          f"std={out_df['water_level'].std():.4f}, "

          f"min={out_df['water_level'].min():.4f}, "

          f"max={out_df['water_level'].max():.4f}")


# --- Also export a simple average ensemble of all 3 ---

ens_pred = np.mean(

    np.column_stack([all_model_preds[m] for m in REFIT_MODELS]),

    axis=1

)

out_ens = test_df[KEY_COLS].copy()

out_ens['water_level'] = ens_pred.astype('float64')

out_ens.to_parquet("submission_m2_nt1_dl_ensemble_top3.parquet", index=False)

print(f"\n  Saved submission_m2_nt1_dl_ensemble_top3.parquet: {out_ens.shape}")

print(f"    water_level: mean={out_ens['water_level'].mean():.4f}, "

      f"std={out_ens['water_level'].std():.4f}")


print("\n=== DONE ===")

print(f"Files created:")

for model_name in REFIT_MODELS:

    print(f"  - submission_m2_nt1_{model_name}.parquet")

print(f"  - submission_m2_nt1_dl_ensemble_top3.parquet")

  Saved submission_m2_nt1_N5_big_tf_w200.parquet: (1422036, 5)
    water_level: mean=39.9006, std=3.2295, min=22.8738, max=47.9106
  Saved submission_m2_nt1_N3_transformer_gcn.parquet: (1422036, 5)
    water_level: mean=39.9224, std=3.2141, min=22.8989, max=48.0087
  Saved submission_m2_nt1_N2_transformer_scse.parquet: (1422036, 5)
    water_level: mean=39.9298, std=3.2279, min=22.9367, max=47.9736

  Saved submission_m2_nt1_dl_ensemble_top3.parquet: (1422036, 5)
    water_level: mean=39.9176, std=3.2225

=== DONE ===
Files created:
  - submission_m2_nt1_N5_big_tf_w200.parquet
  - submission_m2_nt1_N3_transformer_gcn.parquet
  - submission_m2_nt1_N2_transformer_scse.parquet
  - submission_m2_nt1_dl_ensemble_top3.parquet
